This is a markdown cell. It describes the purpose of this notebook and its code.The other cells are coding cells. 

This notebook will extract the information from the scanned Manorial Records and populate excel files with them. 

In [2]:
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 20.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 19.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.5/502.5 kB 10.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 10.7 MB/s eta 0:00:00


In [4]:
!pip install PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.7 MB/s eta 0:00:00a 0:00:01


In [15]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 17.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 17.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 17.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 19.7 MB/s eta 0:00:0000:0100:01


In [35]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.6/239.6 kB 5.2 MB/s eta 0:00:00:00:01


In [20]:
!pip install openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 4.4 MB/s eta 0:00:0000:01


In [1]:
# Import the pandas library with an alias 'pd' (this is a common convention)
import pandas as pd

# Import libraries for pdf reading
from PyPDF2 import PdfReader

# Better use this as it recognizes fonts: 
import pdfplumber

# Import openpyxl to use its functionality for working with Excel files
import openpyxl

# Extracting from word document 
import docx

#Import regular expressionsio
import re

In [5]:
####################### SCRIPT 1  ####################### 


# Function to check if a paragraph is a heading (bold text in this case)
def is_heading(paragraph):
    for run in paragraph.runs:
        if run.bold:
            return True
    return False

def split_into_sentences(paragraph):
   # Replace 's.' with 'shilling' when it follows a number
    paragraph = re.sub(r'(\d)s\.', r'\1 shilling', paragraph)
    # Replace standalone 's.' with 'shilling'
    paragraph = re.sub(r'\bs\.\b', 'shilling', paragraph)
    # Replace 'd.' with 'pence' only when it is preceded by a number (including decimal numbers or with a space)
    paragraph = re.sub(r'(\d+(\.\d+)?\s?)d\.', r'\1pence', paragraph)


    # Replace other abbreviations
    paragraph = paragraph.replace('qrs', 'quarters').replace('bus.', 'bushels')

    # Define a regex pattern for sentence ending
    # Recognize "Total" and "Total," as the beginning of a sentence even without preceding punctuation
    pattern = r'(?<!£)(?<!\s)(?<!\d\s)(?<!\d)[.](?!\d)(?!\s\d)(?!\s[a-z])(?= (?=[A-Z])|(?=$))|(?<=\s)In\s|;|(?=\sTotal\b|(?=\sTotal,))'

    # Split paragraph into sentences based on the updated pattern
    sentences = re.split(pattern, paragraph)

    # Filter out None values and strip sentences
    sentences = [sentence.strip() + (';' if sentence.endswith(';') else '.') for sentence in sentences if sentence and sentence.strip()]

    return sentences

#Extracting monetary amounts
# Updated regular expression pattern for extracting monetary amounts
# Adjusted to capture decimal values before "pence"
def extract_first_monetary_amount(sentence):
    # Initialize default monetary amounts
    monetary_amounts = {'Pounds': 0, 'Shillings': 0, 'Pence': 0}

    # Regex to capture the first coherent monetary group
    monetary_group_pattern = r'£(\d+(?:\.\d+)?)(?:\s(\d+(?:\.\d+)?)\s?shilling)?(?:\s(\d+(?:\.\d+)?)\s?pence)?'
    
    # Try to find a coherent group first
    group_match = re.search(monetary_group_pattern, sentence)
    if group_match:
        monetary_amounts['Pounds'] = float(group_match.group(1) or 0)
        monetary_amounts['Shillings'] = float(group_match.group(2) or 0)
        monetary_amounts['Pence'] = float(group_match.group(3) or 0)
        return monetary_amounts

    # If no coherent group is found, fallback to individual searches (This part is more of a safety net)
    pounds_found = re.search(r'£(\d+(?:\.\d+)?)', sentence)
    shillings_found = re.search(r'(\d+(?:\.\d+)?)\s?shilling', sentence)
    pence_found = re.search(r'(\d+(?:\.\d+)?)\s?pence', sentence)

    if pounds_found:
        monetary_amounts['Pounds'] = float(pounds_found.group(1))
    if shillings_found:
        monetary_amounts['Shillings'] = float(shillings_found.group(1))
    if pence_found:
        monetary_amounts['Pence'] = float(pence_found.group(1))

    return monetary_amounts

document_names = [
    "Adderbury", "Alresford", "Alverstoke", "Ashmansworth", "Beauworth", "Bentley", "Bereleigh", 
    "BishopsFonthill", "BishopsHull", "BishopsSutton", "BishopsWaltham", "Bishopstone", 
    "Bitterne", "Brightwell", "Burghclere", "Cheriton", "Crawley", "Culham", "Downton", 
    "DowntonBorough", "Droxford", "EastKnoyle", "EastMeon", 
    "Ecchinswell", "Esher", "Farnham", "Gosport", "Hambledon", "Harwell",
    "Highclere", "HindonBorough", "Holway", "Ivinghoe", "Merdon", "Morton", "Nailsbourne", "Newtown", "NorthWaltham", 
    "Otterford", "Overton", "OvertonBorough", "Poundisford", "Rimpton", "Southwark",
    "Staplegrove", "StGilesFair", "Taunton", "TauntonBorough", "Twyford", "Upton", 
    "WalthamStLawrence", "Warfield", "Wargrave", "Warren", "WestWycombe", "Wield", "Witney", "WitneyBorough", "Woodhay"
]


# Base paths for the documents and Excel files
doc_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\OCR 1409\\"
excel_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script1\\"

for doc_name in document_names:
    # Construct the full paths for the input and output files
    doc_path = f'{doc_base_path}{doc_name}_1409.docx'
    excel_path = f'{excel_base_path}{doc_name}_1409_1.xlsx'

    # Load the Word Document
    doc = docx.Document(doc_path)

# Load the Word Document (old versions, now a loop through all documents right above)
#doc_path = '/Users/victoriagierok/Dropbox/My Mac (MacBook-Pro.fritz.box)/Desktop/Manorial1301clean.docx'
#doc_path = '/Users/victoriagierok/Dropbox/Physical Capital England/Decennial 2018/Detailed Estimates/Winchester Pipe Rolls/OCR 1301/Taunton.docx'
#doc = docx.Document(doc_path)

    data = []
    current_heading = None

    #Split into columns and calculate total monetary amount
    for para in doc.paragraphs:
        if is_heading(para):
            current_heading = para.text
        else:
            if current_heading:
                sentences = split_into_sentences(para.text)
                for sentence in sentences:
                    if sentence.strip():
                        preprocessed_sentence = sentence.replace('1/2', '.5').replace('1/4', '.25')
    
                        monetary_amounts = extract_first_monetary_amount(preprocessed_sentence)
    
                        total_pounds = float(monetary_amounts['Pounds']) + float(monetary_amounts['Shillings']) / 20 + float(monetary_amounts['Pence']) / 240
                        total_pounds = round(total_pounds, 2)
    
                        data.append({
                            "Heading": current_heading, 
                            "Sentence": preprocessed_sentence,
                            "Total in Pounds": total_pounds, 
                            **monetary_amounts
                        })


    # Create a DataFrame and write to Excel
    df = pd.DataFrame(data)

    # Placeholder DataFrames for additional sheets
    # Replace these with your actual data as needed
    overview_df = pd.DataFrame()  # and so on for other DataFrames...
    
    # Dictionary of DataFrames for each sheet
    dfs = {
        'raw data': df,
        'Overview': overview_df,
        'Receipts': pd.DataFrame(),  # Replace with actual data
        'Expenses': pd.DataFrame(),
        'Issues of the Grange': pd.DataFrame(),
        'Issues of the Mills': pd.DataFrame(),
        'Stock': pd.DataFrame(),
        'Prices': pd.DataFrame(),
        'Labour Rents': pd.DataFrame(),
        'not needed' : pd.DataFrame()
    }

# File path for the Excel file
#excel_path = '/Users/victoriagierok/Dropbox/My Mac (MacBook-Pro.fritz.box)/Desktop/Manorial1301.xlsx'
#excel_path = '/Users/victoriagierok/Dropbox/Physical Capital England/Decennial 2018/Detailed Estimates/Winchester Pipe Rolls/1301-1302/python transcription/script1/Taunton1.xlsx'

    # Using ExcelWriter to write multiple DataFrames to different sheets
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for sheet_name, data in dfs.items():
            data.to_excel(writer, sheet_name=sheet_name, index=False)
    
    print(f"Excel file for {doc_name} created successfully.")




Excel file for Adderbury created successfully.
Excel file for Alresford created successfully.
Excel file for Alverstoke created successfully.
Excel file for Ashmansworth created successfully.
Excel file for Beauworth created successfully.
Excel file for Bentley created successfully.
Excel file for Bereleigh created successfully.
Excel file for BishopsFonthill created successfully.
Excel file for BishopsHull created successfully.
Excel file for BishopsSutton created successfully.
Excel file for BishopsWaltham created successfully.
Excel file for Bishopstone created successfully.
Excel file for Bitterne created successfully.
Excel file for Brightwell created successfully.
Excel file for Burghclere created successfully.
Excel file for Cheriton created successfully.
Excel file for Crawley created successfully.
Excel file for Culham created successfully.
Excel file for Downton created successfully.
Excel file for DowntonBorough created successfully.
Excel file for Droxford created successfu

In [6]:
####################### SCRIPT 2  ####################### 

#This script will sort the different payments into receipts, expenditure, etc.

import pandas as pd
import openpyxl

document_names = [
    "Adderbury", "Alresford", "Alverstoke", "Ashmansworth", "Beauworth", "Bentley", "Bereleigh", 
    "BishopsFonthill", "BishopsHull", "BishopsSutton", "BishopsWaltham", "Bishopstone", 
    "Bitterne", "Brightwell", "Burghclere", "Cheriton", "Crawley", "Culham", "Downton", 
    "DowntonBorough", "Droxford", "EastKnoyle", "EastMeon", 
    "Ecchinswell", "Esher", "Farnham", "Gosport", "Hambledon", "Harwell",
    "Highclere", "HindonBorough", "Holway", "Ivinghoe", "Merdon", "Morton", "Nailsbourne", "Newtown", "NorthWaltham", 
    "Otterford", "Overton", "OvertonBorough", "Poundisford", "Rimpton", "Southwark",
    "Staplegrove", "StGilesFair", "Taunton", "TauntonBorough", "Twyford", "Upton", 
    "WalthamStLawrence", "Warfield", "Wargrave", "Warren", "WestWycombe", "Wield", "Witney", "WitneyBorough", "Woodhay"
]


# Specify the paths to your Excel files
type_excel_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Type.xlsx"

# The loop will dynamically create these paths for each document
input_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script1\\"
output_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script2\\"


# Load the 'Type' Excel file
type_df = pd.read_excel(type_excel_path)

# Create a dictionary to map headings to their types
type_mapping = dict(zip(type_df['Heading'], type_df['Type']))

#new code
for doc_name in document_names:
    # Dynamically construct the specific paths for each document
    manorial_excel_path = f'{input_base_path}{doc_name}_1409_1.xlsx'
    new_excel_path = f'{output_base_path}{doc_name}_1409_2.xlsx'

    # Load the 'raw data' sheet and the entire workbook for each document
    raw_data_df = pd.read_excel(manorial_excel_path, sheet_name='raw data')
    book = openpyxl.load_workbook(manorial_excel_path)


    # Define the column headers
    column_headers = ['Heading', 'Sentence', 'Total in Pounds', 'Pounds', 'Shillings', 'Pence']
    
    # Dictionary to store the data for each sheet, with totals calculation
    sheets_data = {}
    totals = {}  # Dictionary to store totals for each heading
    
    
    # Iterate through each row in the 'raw data' DataFrame
    last_heading = None
    exclude_keywords = ['Total', 'total', 'In total']  # Keywords to exclude in total calculation

    for index, row in raw_data_df.iterrows():
        heading = row['Heading']
        if heading in type_mapping:
            target_sheet_name = type_mapping.get(heading)
            if target_sheet_name:
                if target_sheet_name not in sheets_data:
                    sheets_data[target_sheet_name] = []
    
                # Add an empty row before a new heading
                if last_heading != heading:
                    if last_heading is not None:
                        # Append total for the last heading
                        total_row = {"Heading": last_heading, "Sentence": "Total", "Total in Pounds": totals.get(last_heading, 0)}
                        sheets_data[target_sheet_name].append(total_row)
                        # Append an empty row
                        sheets_data[target_sheet_name].append({})
    
                    last_heading = heading
                    totals[heading] = 0  # Reset total for new heading
                    # Append an empty row
                    sheets_data[target_sheet_name].append({})
    
                # Check if the sentence starts with any of the excluded keywords
                if not any(row['Sentence'].startswith(kw) for kw in exclude_keywords):
                    # Update the total for the current heading
                    totals[heading] += float(row['Total in Pounds'])
    
                # Process and append the current row
                sheets_data[target_sheet_name].append(row)
    
    # Add the total for the last heading in the loop
    if last_heading:
        total_row = {"Heading": last_heading, "Sentence": "Total", "Total in Pounds": totals.get(last_heading, 0)}
        sheets_data[target_sheet_name].append(total_row)


    # Write the aggregated data to the respective sheets in 'Manorial1301.xlsx'
    for sheet_name, data in sheets_data.items():
        sheet_df = pd.DataFrame(data)
    
        # Ensure the workbook has the sheet
        if sheet_name not in book.sheetnames:
            book.create_sheet(sheet_name)
            target_sheet = book[sheet_name]
            target_sheet.append(column_headers)  # Add column headers for a new sheet
        else:
            target_sheet = book[sheet_name]
            if target_sheet.max_row == 1:
                target_sheet.append(column_headers)  # Add column headers if the sheet is empty
    
        # Write data to the sheet
        for index, row in sheet_df.iterrows():
            # Check if the row is not empty (to skip empty rows added for spacing)
            if not row.isnull().all():
                target_sheet.append(row.tolist())


    # After processing, save the workbook to the new path for each document
    book.save(new_excel_path)
    print(f"Processed and saved: {new_excel_path}")


Processed and saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script2\\Adderbury_1409_2.xlsx
Processed and saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script2\\Alresford_1409_2.xlsx
Processed and saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script2\\Alverstoke_1409_2.xlsx
Processed and saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script2\\Ashmansworth_1409_2.xlsx
Processed and saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script2\\Beauworth_1409_2.x

In [7]:
#This works 

####################### SCRIPT 3  ####################### 

################## Creating Error Column ################## 
                        # & #
######### Separating out Bushels and Quarters #############
  
#extract pairs of shil-penc per quart-bush

import pandas as pd
import re
import os 

# Function to ensure the directory exists for the output path
def ensure_dir(file_path):
    directory = os.path.dirname(file_path)
    if not os.path.exists(directory):
        os.makedirs(directory)

# List of document names
document_names = [
    "Adderbury", "Alresford", "Alverstoke", "Ashmansworth", "Beauworth", "Bentley", "Bereleigh", 
    "BishopsFonthill", "BishopsHull", "BishopsSutton", "BishopsWaltham", "Bishopstone", 
    "Bitterne", "Brightwell", "Burghclere", "Cheriton", "Crawley", "Culham", "Downton", 
    "DowntonBorough", "Droxford", "EastKnoyle", "EastMeon", 
    "Ecchinswell", "Esher", "Farnham", "Gosport", "Hambledon", "Harwell",
    "Highclere", "HindonBorough", "Holway", "Ivinghoe", "Merdon", "Morton", "Nailsbourne", "Newtown", "NorthWaltham", 
    "Otterford", "Overton", "OvertonBorough", "Poundisford", "Rimpton", "Southwark",
    "Staplegrove", "StGilesFair", "Taunton", "TauntonBorough", "Twyford", "Upton", 
    "WalthamStLawrence", "Warfield", "Wargrave", "Warren", "WestWycombe", "Wield", "Witney", "WitneyBorough", "Woodhay"
]


# List of all required sheet names in the desired order
required_sheets = [
    "raw data", "Overview", "Receipts", "Expenses", 
    "Issues of the Grange", "Issues of the Mills", 
    "Stock", "Prices", "Labour Rents", "not needed"
]

# Define base paths
input_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script2\\"
output_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script3\\"




#1) Replace abbrevations qrs. for quarters
def replace_abbreviations(sentence):
    sentence = re.sub(r'\bqr\.\b', 'quarters', sentence)
    sentence = re.sub(r'\bqr\b', 'quarters', sentence)
    sentence = re.sub(r'\bbus\b', 'bushels', sentence)  # Add this line
    return sentence

#2) Parse out the type of corn mentioned in each sentence >>> Kuba: Difference from 1301 scripts
def extract_corn_type(sentence):
    sentence_lower = sentence.lower()
    # List of specific corn types to look for
    corn_types = ['wheat', 'mancorn', 'maslin', 'barley', 'first-grade malt', 'second-grade malt', 'peas', 'oats', 'oatmeal', 'vetches', 'rye', 'curall', 'beans', 'dredge', 'malt', 'meal', 'curall']
    #for corn in corn_types:
    #    if re.search(r'\b' + re.escape(corn) + r'\b', sentence_lower):
    #        print(f"Found corn type: {corn} in sentence: {sentence}")
    #        return corn
    #print(f"No corn type found in sentence: {sentence}")
    #return None
    
    # Find corn types in the sentence
    found_corn_types = []
    for corn in corn_types:
        # Use case-insensitive search for each corn type
        if re.search(re.escape(corn), sentence, re.IGNORECASE):
            found_corn_types.append(corn)

    # Return the unique corn types found in the sentence
    return ', '.join(set(found_corn_types))  # Join multiple types with a comma if needed

#3) Extract quantities of quarters and bushels  
def extract_quantities(sentence):
    # Updated pattern to capture pairs of quarters and bushels
    pattern = r'(\d+(?:\.\d+)?)\s?(quarters|bushels)(?:\s?(\d+(?:\.\d+)?)\s?(quarters|bushels))?'
    matches = re.findall(pattern, sentence)

    quantities = []
    for match in matches:
        # Initialize default values for quarters and bushels
        quarters = 0
        bushels = 0

        # Assign values based on the matched groups
        if match[1] == 'quarters':
            quarters = float(match[0])
            if match[3] == 'bushels':
                bushels = float(match[2]) if match[2] else 0
        elif match[1] == 'bushels':
            bushels = float(match[0])
            if match[3] == 'quarters':
                quarters = float(match[2]) if match[2] else 0

        # Append the pair to the quantities list
        quantities.append({'Quarters': quarters, 'Bushels': bushels})

    return quantities

# Example usage
sentence = "£8 10 shilling 6.5pence from 29.5 quarters of wheat sold, at 4 shilling a qr for 5.5quarters, 5 shilling a qr for 2 quarters 5 bushels, 6 shilling a qr for 10 quarters 5 bushels, and, 6 shilling 8pence a qr for 10 quarters 6 bushels."
quantities = extract_quantities(sentence)

#4) Insert empty rows for legibility and calculate error term 
def insert_empty_rows_and_calculate_error(df):
    # Insert an empty row after the first row
    df = pd.concat([df.iloc[:1], pd.DataFrame([['']*len(df.columns)], columns=df.columns), df.iloc[1:]]).reset_index(drop=True)

    # Add a placeholder for the new 'Error' column
    df['Error'] = None

    # Group by 'Heading' and process each group
    grouped = df.groupby('Heading', sort=False)
    new_df = pd.DataFrame()

    for _, group in grouped:
        if len(group) > 1:
            # Calculate the Error for the group (last value - penultimate value in 'Total in Pounds')
            last_value = group['Total in Pounds'].iloc[-1]
            penultimate_value = group['Total in Pounds'].iloc[-2]
            error = last_value - penultimate_value
            # Assign the Error value to the last row of the group
            group.at[group.index[-1], 'Error'] = error

        # Append two empty rows and the group to the new dataframe
        group_with_empty_rows = pd.concat([group, pd.DataFrame([['']*len(df.columns)]*2, columns=df.columns)], ignore_index=True)
        new_df = pd.concat([new_df, group_with_empty_rows], ignore_index=True)

    return new_df

#5) Extracting monetary amounts of per quarter-bushel sales
def extract_monetary_amounts(sentence):
    patterns = {
        'shilling and pence a quarter': r'(\d+(?:\.\d+)?)\s?shilling(?:s)?(?:\s?(\d+(?:\.\d+)?)\s?pence)?\s?a\s?quarter(?:s)?',
        'shilling and pence a bushel': r'((?:\d+(?:\.\d+)?)\s?shilling(?:s)?\s?)?(?:\s?(\d+(?:\.\d+)?)\s?pence)?\s?a\s?bushel(?:s)?'
    }

    # Initialize a dictionary to store results
    results = {key: [] for key in patterns}

    # Iterate through each pattern and find all matches in the sentence
    for key, pattern in patterns.items():
        matches = re.findall(pattern, sentence, re.IGNORECASE)
        for match in matches:
            try:
                # Use regex to extract numeric values only
                shilling = float(re.search(r'(\d+(?:\.\d+)?)', match[0]).group(1)) if match[0] else 0
                pence = float(re.search(r'(\d+(?:\.\d+)?)', match[1]).group(1)) if match[1] else 0
            except (ValueError, AttributeError) as e:
                # Print the problematic match and continue processing
                print(f"Error processing match: {match} in sentence: '{sentence}', Error: {e}")
                shilling = 0
                pence = 0
            results[key].append((shilling, pence))
            
    return results

# Example sentences
example_sentences = [
    "The same render account for £8 10 shilling 6.5pence from 29.5 quarters of wheat sold from the issue of the same mill, at 4 shilling a quarters for 5.5quarters, 5 shilling a quarters for 2 quarters 5 bushels, 6 shilling a quarters for 10 quarters 5 bushels, and, 6 shilling 8pence a quarters for 10 quarters 6 bushels.",
    # Add more sentences here for testing
    #in 1409 they all start with "And for..." 
]

# Test the function
#for sentence in example_sentences:
#    extracted_amounts = extract_monetary_amounts(sentence)
#    print(f"Sentence: {sentence}\nExtracted Amounts: {extracted_amounts}\n")


# Process each document
for doc_name in document_names:
    input_path = f'{input_base_path}{doc_name}_1409_2.xlsx'
    output_path = f'{output_base_path}{doc_name}_1409_3.xlsx'
    ensure_dir(output_path)

    # Check if the input file exists to process
    if not os.path.exists(input_path):
        print(f"File does not exist: {input_path}")
        continue

    # Load the Excel file
    xls = pd.ExcelFile(input_path)
    
    # Process each sheet
    #In the final part of your script where you use pd.ExcelWriter to save processed data, 
    #the variable new_file_path is meant to specify the filename and path where the changes should be saved. 
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for sheet_name in xls.sheet_names:
            df = pd.read_excel(xls, sheet_name)
            if 'Heading' not in df.columns:
                print(f"Skipping '{sheet_name}' in {doc_name}_1409 as it lacks 'Heading' column.")
                continue  # Skip this sheet processing

            
            # Initialize necessary columns to ensure they exist
            for col_name in ['Quarters 1', 'Bushels 1']:
                if col_name not in df.columns:
                    df[col_name] = 0.0  # Initialize with default value


            if sheet_name == 'raw data':
                # Initialize columns for monetary amounts
                monetary_cols = ['shilling a quarter', 'pence a quarter', 'shilling a bushel', 'pence a bushel']
                for col in monetary_cols:
                    for i in range(1, 5):  # Adjust range based on the expected max count
                        df[f'{col} {i}'] = None
    
                for index, row in df.iterrows():
                    # Update 'Sentence' with replaced abbreviations
                    updated_sentence = replace_abbreviations(row['Sentence'])
                    df.at[index, 'Sentence'] = updated_sentence
    
                    # Extract monetary amounts from the updated sentence
                    monetary_amounts = extract_monetary_amounts(updated_sentence)
    
                    # This is the correct handling of the monetary amounts extraction
                    for i, (shilling, pence) in enumerate(monetary_amounts['shilling and pence a quarter']):
                        df.at[index, f'shilling a quarter {i + 1}'] = shilling
                        df.at[index, f'pence a quarter {i + 1}'] = pence
    
                    for i, (shilling, pence) in enumerate(monetary_amounts['shilling and pence a bushel']):
                        df.at[index, f'shilling a bushel {i + 1}'] = shilling
                        df.at[index, f'pence a bushel {i + 1}'] = pence
    
                    #print(f"Processing Updated Sentence: {updated_sentence}")  # Debugging print
                    monetary_amounts = extract_monetary_amounts(updated_sentence)
                    #print("Monetary Amounts:", monetary_amounts)  # Debugging print

            #print(df[['shilling a quarter 1', 'shilling a quarter 2', 'shilling a bushel 1', 'shilling a bushel 2']].head())  # Adjust column names as per your DataFrame


            #for col in monetary_cols:
            #   for i, value in enumerate(monetary_amounts[col]):
            #        col_name = f'{col} {i + 1}'
            #       print(f"Assigning {value} to column {col_name}")  # Debugging print
            #       if col_name in df.columns:
            #           df.at[index, col_name] = value
      
                # Add new columns for 'Corn', 'Quarters', and 'Bushels'
                df['Corn'] = None
                max_pairs = 0
                for index, row in df.iterrows():
                    # Update 'Sentence' with replaced abbreviations
                    updated_sentence = replace_abbreviations(row['Sentence'])
                    df.at[index, 'Sentence'] = updated_sentence
                    
                    # Extract and update 'Corn' types
                    corn_type = extract_corn_type(updated_sentence)
                    df.at[index, 'Corn'] = corn_type
    
                    # Extract quantities and update 'Quarters' and 'Bushels'
                    quantities = extract_quantities(updated_sentence)
                    max_pairs = max(max_pairs, len(quantities))
                    for i, quantity_pair in enumerate(quantities):
                        df.at[index, f'Quarters {i+1}'] = quantity_pair['Quarters']
                        df.at[index, f'Bushels {i+1}'] = quantity_pair['Bushels']
    
                # Add alternating columns for quarters and bushels with float dtype
                for i in range(max_pairs):
                    if f'Quarters {i+1}' not in df.columns:
                        df[f'Quarters {i+1}'] = pd.Series(dtype='float')
                    if f'Bushels {i+1}' not in df.columns:
                        df[f'Bushels {i+1}'] = pd.Series(dtype='float')

                # Initialize new columns for total quarters and bushels
                df['Total Quarters Except 1'] = 0.0
                df['Total Bushels Except 1'] = 0.0
                df['Total Corn Cross-Check'] = 0.0
                df['Total Corn Manorial Account'] = 0.0
                df['Corn Quantity Error'] = 0.0
    
                for index, row in df.iterrows():
                    total_quarters = 0.0
                    total_bushels = 0.0
    
                   # Sum up quarters and bushels from 2nd column onwards
                    for i in range(2, max_pairs + 1):
                        quarters_col = f'Quarters {i}'
                        bushels_col = f'Bushels {i}'
    
                       # Check if the column exists and add its value if it does
                        if quarters_col in df.columns:
                            total_quarters += row[quarters_col] if not pd.isna(row[quarters_col]) else 0.0
                        if bushels_col in df.columns:
                            total_bushels += row[bushels_col] if not pd.isna(row[bushels_col]) else 0.0
    
                    # Assign Quarter 1 and Bushel 1 to totals if no other quarters and bushels exist
                    #if total_quarters == 0 and total_bushels == 0 and 'Quarters 1' in df.columns and 'Bushels 1' in df.columns:
                    #    total_quarters = row['Quarters 1'] if not pd.isna(row['Quarters 1']) else 0.0
                    #    total_bushels = row['Bushels 1'] if not pd.isna(row['Bushels 1']) else 0.0


                    # Assign Quarter 1 and Bushel 1 to totals if no other quarters and bushels exist
                    if 'Quarters 1' in df.columns and 'Bushels 1' in df.columns:
                        if total_quarters == 0:
                            total_quarters = row['Quarters 1'] if not pd.isna(row['Quarters 1']) else 0.0
                        if total_bushels == 0:
                            total_bushels = row['Bushels 1'] if not pd.isna(row['Bushels 1']) else 0.0


                    df.at[index, 'Total Quarters Except 1'] = total_quarters
                    df.at[index, 'Total Bushels Except 1'] = total_bushels
                    df.at[index, 'Total Corn Cross-Check'] = total_quarters + (total_bushels / 8)
                    df.at[index, 'Total Corn Manorial Account'] = (row['Quarters 1'] if 'Quarters 1' in df.columns and not pd.isna(row['Quarters 1']) else 0) + \
                                                                  ((row['Bushels 1'] if 'Bushels 1' in df.columns and not pd.isna(row['Bushels 1']) else 0) / 8)
                    df.at[index, 'Corn Quantity Error'] = df.at[index, 'Total Corn Cross-Check'] - df.at[index, 'Total Corn Manorial Account']

                    
                    # Store the sums in the new columns
                    #df.at[index, 'Total Quarters Except 1'] = total_quarters
                    #df.at[index, 'Total Bushels Except 1'] = total_bushels

                   # Calculate Total Corn Cross-Check and Total Corn Manorial Account
                   # df.at[index, 'Total Corn Cross-Check'] = total_quarters + (total_bushels / 8)
                   # quarters_1 = row['Quarters 1'] if not pd.isna(row['Quarters 1']) else 0.0
                   # bushels_1 = row['Bushels 1'] if not pd.isna(row['Bushels 1']) else 0.0
                   # df.at[index, 'Total Corn Manorial Account'] = quarters_1 + (bushels_1 / 8)
    
                    # Calculate Corn Quantity Error
                    cross_check_total = df.at[index, 'Total Corn Cross-Check']
                    manorial_account_total = df.at[index, 'Total Corn Manorial Account']
                    df.at[index, 'Corn Quantity Error'] = cross_check_total - manorial_account_total
    
    
                # Define the initial set of columns
                ordered_columns = ['Heading', 'Sentence', 'Total in Pounds', 'Pounds', 'Shillings', 'Pence', 'Corn', 
                       'Total Quarters Except 1', 'Total Bushels Except 1', 'Total Corn Cross-Check', 
                       'Total Corn Manorial Account', 'Corn Quantity Error']


                # Determine the number of 'Quarters' and 'Bushels' columns
                num_quarters_bushels_columns = max([int(col.split(' ')[-1]) for col in df.columns if 'Quarters' in col or 'Bushels' in col], default=0)
    
                # Add 'Quarters' and 'Bushels' columns in pairs
                for i in range(1, num_quarters_bushels_columns + 1):
                    ordered_columns.append(f'Quarters {i}')
                    ordered_columns.append(f'Bushels {i}')
    
                # Determine the number of 'shilling a quarter' and 'pence a quarter' columns
                num_quarter_columns = max([int(col.split(' ')[-1]) for col in df.columns if 'shilling a quarter' in col or 'pence a quarter' in col], default=0)
    
                # Add 'shilling a quarter' and 'pence a quarter' columns in pairs
                for i in range(1, num_quarter_columns + 1):
                    ordered_columns.append(f'shilling a quarter {i}')
                    ordered_columns.append(f'pence a quarter {i}')
    
                # Determine the number of 'shilling a bushel' and 'pence a bushel' columns
                num_bushel_columns = max([int(col.split(' ')[-1]) for col in df.columns if 'shilling a bushel' in col or 'pence a bushel' in col], default=0)
    
                # Add 'shilling a bushel' and 'pence a bushel' columns in pairs
                for i in range(1, num_bushel_columns + 1):
                    ordered_columns.append(f'shilling a bushel {i}')
                    ordered_columns.append(f'pence a bushel {i}')
    
                # Reorder the DataFrame
                df = df[ordered_columns]


            elif sheet_name in ['Receipts', 'Expenses']:
                # Process 'Receipts' and 'Expenses' sheets with your error calculation
                df = insert_empty_rows_and_calculate_error(df)


            # Example of safely accessing the columns
            if 'Quarters 1' in df.columns and 'Bushels 1' in df.columns:
            # Perform operations with these columns
                pass

            # Save each processed sheet to the new Excel file
            df.to_excel(writer, sheet_name=sheet_name, index=False)


        # At the end of the Excel writing block (right before closing the writer context)
        for sheet_name in required_sheets:
            if sheet_name not in writer.book.sheetnames:
                # Create an empty DataFrame
                empty_df = pd.DataFrame()
                # Save it as an empty sheet
                empty_df.to_excel(writer, sheet_name=sheet_name, index=False)

         # Ensure there is an active sheet at the end of processing
        if writer.book.sheetnames:
            # Set the first sheet in the workbook as active
            first_sheet = writer.book[writer.book.sheetnames[0]]
            writer.book.active = first_sheet
    
        print(f"Processed and saved: {output_path}")






Skipping 'Overview' in Adderbury_1409 as it lacks 'Heading' column.
Skipping 'Issues of the Mills' in Adderbury_1409 as it lacks 'Heading' column.
Skipping 'Prices' in Adderbury_1409 as it lacks 'Heading' column.
Skipping 'Labour Rents' in Adderbury_1409 as it lacks 'Heading' column.
Skipping 'not needed' in Adderbury_1409 as it lacks 'Heading' column.
Processed and saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script3\\Adderbury_1409_3.xlsx
Skipping 'Overview' in Alresford_1409 as it lacks 'Heading' column.
Skipping 'Issues of the Mills' in Alresford_1409 as it lacks 'Heading' column.
Skipping 'Prices' in Alresford_1409 as it lacks 'Heading' column.
Skipping 'Labour Rents' in Alresford_1409 as it lacks 'Heading' column.
Skipping 'not needed' in Alresford_1409 as it lacks 'Heading' column.
Processed and saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2

In [14]:
    ####################### SCRIPT 4  ####################### 
    
    # This script will sort Issues of the Grange Quantities
    # Goal 1: Sort out how much corn of each variety was produced vs. sold vs. sewn. etc.
    # Goal 2: Find out the value of all the corn produced! 
    
    
####################### BASIC SETUP #######################
import pandas as pd
import re

# List of document names
# List of document names
document_names = [
    "Adderbury", "Alresford", "Alverstoke", "Ashmansworth", "Beauworth", "Bentley", "Bereleigh", 
    "BishopsFonthill", "BishopsSutton", "BishopsWaltham", "Bishopstone", 
    "Bitterne", "Brightwell", "Burghclere", "Cheriton", "Crawley", "Culham", "Downton", 
    "DowntonBorough", "Droxford", "EastKnoyle", "EastMeon", 
    "Ecchinswell", "Esher", "Farnham", "Gosport", "Hambledon", "Harwell",
    "Highclere", "HindonBorough", "Holway", "Ivinghoe", "Merdon", "Morton", "Newtown", "NorthWaltham", 
    "Otterford", "OvertonBorough", "Poundisford", "Rimpton", "Southwark",
    "Staplegrove", "StGilesFair", "Taunton", "TauntonBorough", "Twyford", "Upton", 
    "WalthamStLawrence", "Warfield", "Wargrave", "Warren", "WestWycombe", "Wield", "Witney", "WitneyBorough", "Woodhay"
]
    

# Directory path
base_input_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script3\\"
base_output_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script4\\"

for doc_name in document_names: #Kuba: Changed indent to 1-lvl lower
    print(f"\nProcessing document: {doc_name}")
    input_file = f"{base_input_path}{doc_name}_1409_3.xlsx"
    output_file = f"{base_output_path}{doc_name}_1409_4.xlsx"

    ####################### IMPORT & DATAFRAME DEFINITION #######################
    # Load all sheets into a dictionary of DataFrames
    print(f"Loading Excel file: {input_file}")
    all_sheets_df = pd.read_excel(input_file, sheet_name=None)
        
    
    #I am basically isolating the Issues of the Grange sheet as its own dataframe; same for raw data
    issues_df = all_sheets_df['Issues of the Grange']
    raw_data_df = all_sheets_df['raw data']


    # Check if 'Issues of the Grange' is in the dictionary and if 'Sentence' is a column in the DataFrame
    if 'Issues of the Grange' in all_sheets_df and 'Sentence' in all_sheets_df['Issues of the Grange'].columns:
            issues_df = all_sheets_df['Issues of the Grange']
        
            ####################### COLUMNS ADJUSTED AND COPIED FROM RAW DATA #######################
            #All the action now happens on the issues_df which is the Issues of the Grange dataframe
            # Delete unwanted columns from 'Issues of the Grange'
            columns_to_delete = ["Total in Pounds", "Pounds", "Shillings", "Pence"]
            issues_df.drop(columns=columns_to_delete, inplace=True, errors='ignore')
            
            # Dynamically construct the columns to be copied from 'raw data'
            columns_to_copy = ["Total Quarters Except 1", "Total Bushels Except 1", 
                                "Total Corn Cross-Check", "Total Corn Manorial Account", 
                                "Corn Quantity Error"]

            ####################### EXTRACTING RELEVANT QUANTITIES BASED ON KEYWORDS #######################
            # Function to extract quarters and bushels based on specific keywords
            def extract_specific_quarters_bushels(sentence, keywords):
                # Convert sentence to lowercase for case-insensitive matching
                sentence_lower = sentence.lower()
            
                # Check if any keyword is in the sentence
                if any(keyword in sentence_lower for keyword in keywords):
                    # Regular expression to find patterns like 'xx quarters' or 'xx quarters yy bushels'
                    matches = re.findall(r'(\d+(?:\.\d+)?)\s+quarters(?:\s+(\d+(?:\.\d+)?)\s+bushels)?', sentence, re.IGNORECASE)
                    if matches:
                        # Summing up all found quantities
                        total_quarters = sum(float(quarter) for quarter, bushel in matches if quarter)
                        total_bushels = sum(float(bushel) if bushel else 0 for quarter, bushel in matches)
                        return total_quarters, total_bushels
                return 0, 0
        
            # Categories and their respective keywords
            categories = {
                'Sown': ['sown'],
                'Bought in': ['bought'],
                'Tithe': ['tithe'],
                'Manorial servants & livestock': ['delivered to the door-keeper', 'provision of the steward', 
                                                    'provision of the constable', 'for the liveries of', 'fodder of'],
                'Sold': ['sold'],
                'Account': ['same render account', 'from the whole issue', 'same renders account']  # Add this line
            }
        
            # Create columns for each category to ensure they exist even if no keywords are found
            for category in categories:
                issues_df[f'{category} Quarters'] = 0
                issues_df[f'{category} Bushels'] = 0
                issues_df[f'{category} Total Quarters'] = 0
        
        
        
            # Process each category (sown, tithe, bought, etc.) with correct quantities
            for category, keywords in categories.items():
                # Process each sentence to extract quarters and bushels, ensuring we handle the case where no data is found
                category_data = issues_df['Sentence'].apply(lambda x: extract_specific_quarters_bushels(x, keywords))
                issues_df[f'{category} Quarters'], issues_df[f'{category} Bushels'] = zip(*category_data)
                issues_df[f'{category} Total Quarters'] = issues_df[f'{category} Quarters'] + issues_df[f'{category} Bushels'] / 8
            
                # Fill NaNs right here to handle cases where no data might have led to NaNs
                issues_df[f'{category} Quarters'].fillna(0, inplace=True)
                issues_df[f'{category} Bushels'].fillna(0, inplace=True)
                issues_df[f'{category} Total Quarters'].fillna(0, inplace=True)
            
            # Print output to check columns created in issues_df
            print("Check1: Columns in issues_df after initialization:", issues_df.columns)
        
            # Extra processing step for account category
            # Check for both "bought" and "same render account" in the sentence for the "Account" category
            if category == 'Account':
                issues_df['bought only'] = issues_df['Sentence'].apply(lambda sentence: 1 if ('bought' in sentence.lower() and "same render account" in sentence.lower()) else 0)
            
            # Loop through each category and apply the extraction logic
            for category, keywords in categories.items():
                # Extracting data for the current category
                category_data = issues_df['Sentence'].apply(lambda x: extract_specific_quarters_bushels(x, keywords))
            
                # Unpack the data and add to DataFrame
                issues_df[f'{category} Quarters'], issues_df[f'{category} Bushels'] = zip(*category_data)
                # Convert all bushels to quarters for total
                issues_df[f'{category} Total Quarters'] = issues_df[f'{category} Quarters'] + issues_df[f'{category} Bushels'] / 8
            
            
            # List of corn types
            corn_types = ['wheat', 'mancorn', 'maslin', 'barley', 'first-grade malt', 'second-grade malt', 'peas', 'oats', 'oatmeal', 'vetches', 'rye', 'dredge', 'malt', 'meal', 'curall']
            
            
            # Append "Quarters x" and "Bushels x" where x is from 1 to 30
            for i in range(1, 31):
                quarter_column = f"Quarters {i}"
                bushel_column = f"Bushels {i}"
                if quarter_column in raw_data_df.columns:
                    columns_to_copy.append(quarter_column)
                if bushel_column in raw_data_df.columns:
                    columns_to_copy.append(bushel_column)

            ####################### MERGE ISSUES_DF WITH RAW DATA #######################
            # Merge 'Issues of the Grange' with the relevant columns from 'raw data' based on "Heading" and "Sentence"
            #these are: columns_to_copy = ["Total Quarters Except 1", "Total Bushels Except 1", "Total Corn Cross-Check", "Total Corn Manorial Account", "Corn Quantity Error"]
            # Only include columns from raw_data_df that actually exist
            all_columns = ["Heading", "Sentence"] + columns_to_copy 

            # Strip whitespace and lowercase the columns before merging
            issues_df['Heading'] = issues_df['Heading'].str.strip().str.lower()
            raw_data_df['Heading'] = raw_data_df['Heading'].str.strip().str.lower()
            issues_df['Sentence'] = issues_df['Sentence'].str.strip().str.lower()
            raw_data_df['Sentence'] = raw_data_df['Sentence'].str.strip().str.lower()

            #Add debugging output before and after critical operations to better understand where the data is being lost or causing issues:
            print("Shape of issues_df before merge:", issues_df.shape)
            print("Shape of raw_data_df:", raw_data_df.shape)

            issues_df = issues_df.merge(raw_data_df[all_columns], on=["Heading", "Sentence"], how="left")

            print("Shape of issues_df after merge:", issues_df.shape)

            if issues_df.empty:
                print(f"issues_df is empty after merging for {doc_name}, skipping file save.")
                #continue  # Skip to the next document >>> Kuba: Commented to ensure succesful creation of docs even if sheet missing. Allows continuation into script 5

            # After merging, ensure that dynamically created columns like "Sold Total Quarters" are still present
            # List all dynamically created columns that should be in issues_df
            dynamically_created_columns = [
                'Sown Quarters', 'Sown Bushels', 'Sown Total Quarters',
                'Bought in Quarters', 'Bought in Bushels', 'Total Bought Quarters',
                'Tithe Quarters', 'Tithe Bushels', 'Tithe Total Quarters',
                'Manorial servants & livestock Quarters', 'Manorial servants & livestock Bushels', 'Manorial servants & livestock Total Quarters',
                'Sold Quarters', 'Sold Bushels', 'Sold Total Quarters',  # Including your 'Sold Total Quarters'
                'Account Quarters', 'Account Bushels', 'Account Total Quarters',
                'bought only'
            ]

            # Ensure that all dynamically created columns are present after the merge
            for col in dynamically_created_columns:
                if col not in issues_df.columns:
                    print(f"Column '{col}' not found in 'issues_df' after merge. Adding it back with default value 0.")
                    issues_df[col] = 0  # or pd.NA if you prefer NaN

            
            ####################### SAVE FINAL VERSION AND RENAME TO ISSUES GRANGE RAW #######################
            # Attach updated issues_df ('Issues of the Grange') back to the all_sheets_df
            # By using the .copy() method, you make sure that the df stored under "Issues of the Grange raw" is a separate copy from any further manipulations you might apply to issues_df after renaming it in the dictionary.
            all_sheets_df['Issues of the Grange'] = issues_df.copy()
            # Rename 'Issues of the Grange' to 'Issues of the Grange raw' in the all_sheets_df dictionary
            all_sheets_df['Issues of the Grange raw'] = all_sheets_df.pop('Issues of the Grange')
            print("Updated columns after processing: ", all_sheets_df['Issues of the Grange raw'].columns)
            
            
    
            ##################################################################################################
            #        GOAL 1: Sort out how much corn of each variety was produced vs. sold vs. sewn. etc.
            ##################################################################################################
            
            ####################### AGGREGATE INFORMATION IN ISSUES #######################
            # Step 1: Aggregate 'bought only' information - in the old issues_df
            bought_only_aggregation = issues_df.groupby('Heading')['bought only'].max().reset_index()
            bought_only_aggregation.rename(columns={'bought only': 'Bought Only Aggregated'}, inplace=True)
            
            # Convert this into a dictionary for easier mapping:
            bought_only_dict = bought_only_aggregation.set_index('Heading')['Bought Only Aggregated'].to_dict()
        
            ####################### NEW DICTIONARY CREATION #######################
            #Step 2: New df creation: Initialize the new DataFrame with appropriate data from issues_df
            ### Okay so here it gets interesting with the dataframes
            #I am creating a completely new and empty dataframe called new_sheets_df 
            #which I then attach to the full excel sheet dataframe with all the sheets
            #to do this I start with a list of dictionaries 
            ####################### ENSURE ALL COLUMNS EXIST #######################
            # Ensure that all potential columns are created in issues_df, filled with zeros if no data was found
            for category in categories:
                quarter_col = f'{category} Quarters'
                bushel_col = f'{category} Bushels'
                total_quarter_col = f'{category} Total Quarters'
                if quarter_col not in issues_df.columns:
                    issues_df[quarter_col] = 0
                if bushel_col not in issues_df.columns:
                    issues_df[bushel_col] = 0
                if total_quarter_col not in issues_df.columns:
                    issues_df[total_quarter_col] = 0
    
    
            ####################### CREATE NEW DATAFRAME FROM AGGREGATED DATA #######################
            # Initialize new_rows list for creating new_sheet_df
            new_rows = []
            
            # Populate new_rows with data aggregated based on headings and specific conditions
            for idx, row in issues_df.iterrows():
                new_row = {
                    'Gross Output': row['Heading'],
                    'Account Total Quarters': row.get('Account Total Quarters', 0),
                    'Bought Only Aggregated': bought_only_dict.get(row['Heading'], 0),
                    'Total Bought Quarters': row.get('Total Bought Quarters', 0),
                    'Sold Total Quarters': row.get('Sold Total Quarters', 0)
                }
                new_rows.append(new_row)
    

            ####################### POPULATE DICTIONARIES & CREATE NEW DATAFRAME #######################
            # Populate the new dictionaries with aggregated rows for each corn type
            # Add rows for each corn type
            for corn_type in corn_types:
                bought_only_value = bought_only_aggregation.loc[bought_only_aggregation['Heading'].str.lower() == corn_type.lower(), 'Bought Only Aggregated'].max()
                new_row = {
                    'Gross Output': corn_type.capitalize(),
                    'Bought Only Aggregated': bought_only_value if pd.notnull(bought_only_value) else 0,
                    'Account Total Quarters': 0,
                    'Total Bought Quarters': 0,
                    'Sold Total Quarters': 0
                }
                new_rows.append(new_row)
        
            #creates a new DataFrame called new_sheet_df from a list of dictionaries stored in new_rows
            new_sheet_df = pd.DataFrame(new_rows)

            # Explicitly cast relevant columns to float
            numerical_columns = [
                'Account Total Quarters', 'Bought Only Aggregated', 'Total Bought Quarters', 'Sold Total Quarters',
                'Account Quarters', 'Account Bushels', 'Bought in Quarters', 'Bought in Bushels', 
                'Sold Quarters', 'Sold Bushels', 'Difference Produced+Bought-Sold'
            ]
            
            # Ensure the columns exist and cast them to float
            for col in numerical_columns:
                if col in new_sheet_df.columns:
                    new_sheet_df[col] = new_sheet_df[col].astype(float)

            # Ensure all necessary columns exist before processing
            columns_to_ensure = [
                'Account Total Quarters', 'Bought Only Aggregated', 'Total Bought Quarters', 'Sold Total Quarters',
                'Account Quarters', 'Account Bushels', 'Bought in Quarters', 'Bought in Bushels', 
                'Sold Quarters', 'Sold Bushels', 'Difference Produced+Bought-Sold'
            ]
            
            # Check and add any missing columns with default values
            # Check and add any missing columns with default values
            for col in columns_to_ensure:
                if col not in new_sheet_df.columns:
                    print(f"Column '{col}' not found in 'new_sheet_df'. Adding it with default value 0.")
                    new_sheet_df[col] = 0  # Adding column with default value 0 if it does not exist
            


            # Fill NaN values with 0 in the relevant columns before performing calculations
            # This step will only affect cells with NaN values and will not overwrite any valid existing numbers
            new_sheet_df['Account Total Quarters'] = new_sheet_df['Account Total Quarters'].fillna(0)
            new_sheet_df['Total Bought Quarters'] = new_sheet_df['Total Bought Quarters'].fillna(0)
            new_sheet_df['Sold Total Quarters'] = new_sheet_df['Sold Total Quarters'].fillna(0)

            
            # By assigning new_sheet_df to all_sheets_df['Issues of the Grange'], 
            #you update the dictionary so that the key 'Issues of the Grange' now points to the new DataFrame. 
            all_sheets_df['Issues of the Grange'] = new_sheet_df
            
            
            # Specified columns to extract non-zero values from
            specified_columns = [
                "Account Quarters", "Account Bushels", "Account Total Quarters",
                "Total Bought Quarters", "Bought in Bushels",
                "Sold Quarters",	"Sold Bushels",	"Sold Total Quarters"
            ]
            
            ####################### UPDATE COLUMNS #######################
            #This line would effectively update the 'Total Sold Quarters' column (or whichever is specified by column) 
            #in rows where the 'Gross Output' matches the specified corn_type with the non_zero_value.
            
            ### Kuba: Line appears to cause a zero value error -> Commented out for now
            # all_sheets_df['Issues of the Grange'].loc[all_sheets_df['Issues of the Grange']['Gross Output'].str.lower() == corn_type, column] = non_zero_value
            ### / 
        
            """ ### Kuba: Ultra low effort fix that worked in 1301 script
                    # Ensure key columns are string-typed and normalized -> Needed to prevent errors in case previous cells mess up
            new_sheet_df['Gross Output'] = new_sheet_df['Gross Output'].astype(str).str.strip()
            issues_df['Heading'] = issues_df['Heading'].astype(str).str.strip()

            for corn_type in corn_types:
                mask = new_sheet_df['Gross Output'].str.lower() == corn_type.lower()

                for column in specified_columns:
                    # Check target column exists
                    if column not in new_sheet_df.columns:
                        new_sheet_df[column] = 0.0

                    # If source column missing in issues_df, skip
                    if column not in issues_df.columns:
                        continue

                    # Take the max non-zero value for this corn_type/column
                    src = pd.to_numeric(
                        issues_df.loc[issues_df['Heading'].str.lower() == corn_type.lower(), column],
                        errors='coerce'
                    )
                    value = src.where(src != 0).max()

                    # If real value found, set it
                    if pd.notna(value):
                        new_sheet_df.loc[mask, column] = float(value)

            # Write back
            all_sheets_df['Issues of the Grange'] = new_sheet_df
            ### /

            for corn_type in corn_types:
                corn_row = issues_df[issues_df['Heading'].str.lower() == corn_type.lower()] #This line filters issues_df to find all rows where the 'Heading' matches the current corn_type in the loop, after converting both to lowercase for case-insensitive comparison. The result is stored in corn_row, which will be a DataFrame containing all rows that match the corn type.
                for column in specified_columns: #This nested loop iterates through a list of columns specified in specified_columns, before applying code below
                    non_zero_value = corn_row[corn_row[column] != 0][column].max()  #this finds the largest non-zero entry for the current corn type and column
                    if pd.notna(non_zero_value): #This checks if the non_zero_value is not NaN (i.e., it's a valid number). If true, the following line executes:
                        # Ensure we're updating the correct row in 'Issues of the Grange'
                        new_sheet_df.loc[new_sheet_df['Gross Output'].str.lower() == corn_type.lower(), column] = non_zero_value #This line updates new_sheet_df by setting the value in the specified column for rows where the 'Gross Output' matches the current corn type
            
            
            #Before performing operations on new_sheet_df, explicitly check for column existence:
            print("Columns available in new_sheet_df:", new_sheet_df.columns)
            if 'Bought in Total Quarters' not in new_sheet_df.columns:
                print("Error: 'Bought in Total Quarters' not found in new_sheet_df.")
                # Optionally add the column if it's critical for subsequent operations
                new_sheet_df['Bought in Total Quarters'] = 0
            else:
                new_sheet_df['Bought in Total Quarters'] = new_sheet_df['Bought in Total Quarters'].fillna(0)
            
        
            #Is this still needed?
            # Fill NaN values with 0 in the relevant columns before performing calculations
            new_sheet_df['Account Total Quarters'] = new_sheet_df['Account Total Quarters'].fillna(0)
            new_sheet_df['Bought in Total Quarters'] = new_sheet_df['Bought in Total Quarters'].fillna(0)
            new_sheet_df['Sold Total Quarters'] = new_sheet_df['Sold Total Quarters'].fillna(0)
        
        
            ####################### CALCULATE TOTAL CORN QUANTITY #######################
            #Step 4: 
            # Add 'Total Corn Quantity' column by summing 'Account Total Quarters' and 'Bought in Total Quarters'
            # Modify the 'Total Corn Quantity' calculation based on the condition
            def calculate_total_corn_quantity(row):
                if row['Bought Only Aggregated'] == 1:  # Assuming 'Bought Only Aggregated' is the correct column name after merging
                    # If 'bought only' is 1, do not include 'Bought in Total Quarters' in the sum
                    return row['Account Total Quarters']
                else:
                    # Else, sum 'Account Total Quarters' and 'Bought in Total Quarters'
                    return row['Account Total Quarters'] + row['Bought in Total Quarters']
            
            # Apply the function to each row
            new_sheet_df['Total Corn Quantity'] = new_sheet_df.apply(calculate_total_corn_quantity, axis=1)
            
            # Calculate 'Difference Produced+Bought-Sold' by subtracting 'Sold Total Quarters' from 'Total Corn Quantity'
            new_sheet_df['Difference Produced+Bought-Sold'] = new_sheet_df['Total Corn Quantity'] - new_sheet_df['Sold Total Quarters']
            
            
            # Make sure all expected columns exist, add them if they don't
            new_column_order = [
                'Gross Output', 'Bought Only Aggregated', 'Account Quarters', 'Account Bushels', 
                'Account Total Quarters', 'Bought in Quarters', 'Bought in Bushels', 
                'Bought in Total Quarters', 'Sold Quarters', 'Sold Bushels', 
                'Sold Total Quarters', 'Total Corn Quantity', 'Difference Produced+Bought-Sold'
            ]
        
            for column in new_column_order:
                if column not in new_sheet_df.columns:
                    new_sheet_df[column] = 0  # or appropriate default value
            
            # Reorder the DataFrame columns before saving
            new_sheet_df = new_sheet_df[new_column_order]
            
            # Make sure to replace `new_sheet_df` in the dictionary after updates
            all_sheets_df['Issues of the Grange'] = new_sheet_df
            # / """
    
            ####################### FILL IN NEW DATAFRAME WITH AGGREGATE INFORMATION FROM ISSUES #######################
            #Step 3: Only copying over aggregated data from issues_df to new_sheet
            #The purpose is to populate new_sheet_df with the maximum non-zero values from issues_df 
            #for each specified corn type and column. 
            #This is typically used to summarize data from a detailed DataFrame into a more aggregated form 
            for corn_type in corn_types:
                corn_row = issues_df[issues_df['Heading'].str.lower() == corn_type.lower()]
                for column in specified_columns:
                    non_zero_value = corn_row[corn_row[column] != 0][column].max()
                    if pd.notna(non_zero_value):
                        # Convert non_zero_value to float to ensure compatibility with column dtype
                        new_sheet_df[column] = new_sheet_df[column].astype(float)  # Cast to float
                        new_sheet_df.loc[new_sheet_df['Gross Output'].str.lower() == corn_type.lower(), column] = float(non_zero_value)
            
            #Before performing operations on new_sheet_df, explicitly check for column existence:
            print("Columns available in new_sheet_df:", new_sheet_df.columns)
            if 'Total Bought Quarters' not in new_sheet_df.columns:
                print("Error: 'Total Bought Quarters' not found in new_sheet_df.")
                # Optionally add the column if it's critical for subsequent operations
                new_sheet_df['Total Bought Quarters'] = 0
            else:
                new_sheet_df['Total Bought Quarters'] = new_sheet_df['Total Bought Quarters'].fillna(0)
            
        
            #Is this still needed?
            # Fill NaN values with 0 in the relevant columns before performing calculations
            new_sheet_df['Account Total Quarters'] = new_sheet_df['Account Total Quarters'].fillna(0)
            new_sheet_df['Total Bought Quarters'] = new_sheet_df['Total Bought Quarters'].fillna(0)
            new_sheet_df['Sold Total Quarters'] = new_sheet_df['Sold Total Quarters'].fillna(0)
        
        
            ####################### CALCULATE TOTAL CORN QUANTITY #######################
            #Step 4: 
            # Add 'Total Corn Quantity' column by summing 'Account Total Quarters' and 'Bought in Total Quarters'
            # Modify the 'Total Corn Quantity' calculation based on the condition
            def calculate_total_corn_quantity(row):
                if row['Bought Only Aggregated'] == 1:  # Assuming 'Bought Only Aggregated' is the correct column name after merging
                    # If 'bought only' is 1, do not include 'Bought in Total Quarters' in the sum
                    return row['Account Total Quarters']
                else:
                    # Else, sum 'Account Total Quarters' and 'Bought in Total Quarters'
                    return row['Account Total Quarters'] + row['Total Bought Quarters']
            
            # Apply the function to each row
            new_sheet_df['Total Corn Quantity'] = new_sheet_df.apply(calculate_total_corn_quantity, axis=1)
            
            # Calculate 'Difference Produced+Bought-Sold' by subtracting 'Sold Total Quarters' from 'Total Corn Quantity'
            new_sheet_df['Difference Produced+Bought-Sold'] = new_sheet_df['Total Corn Quantity'] - new_sheet_df['Sold Total Quarters']
            
            
            # Make sure all expected columns exist, add them if they don't
            new_column_order = [
                'Gross Output', 'Bought Only Aggregated', 'Account Quarters', 'Account Bushels', 
                'Account Total Quarters', 'Bought in Quarters', 'Bought in Bushels', 
                'Total Bought Quarters', 'Sold Quarters', 'Sold Bushels', 
                'Sold Total Quarters', 'Total Corn Quantity', 'Difference Produced+Bought-Sold'
            ]
        
            for column in new_column_order:
                if column not in new_sheet_df.columns:
                    new_sheet_df[column] = 0  # or appropriate default value
            
            # Reorder the DataFrame columns before saving
            new_sheet_df = new_sheet_df[new_column_order]
            
            # Make sure to replace `new_sheet_df` in the dictionary after updates
            all_sheets_df['Issues of the Grange'] = new_sheet_df 
        
    
    
    
            ##################################################################################################
            #        GOAL 2: Extract sale of corn values and calculate average price per qrs
            ##################################################################################################
            
            #After extracting sale of corn values in a separate df, import into Issue of the Grange 
            #raw_data_df has been defined above, it's just the raw data sheet
            sale_of_corn_prices_df = raw_data_df[raw_data_df['Heading'].str.contains(
                "Sale of Corn|Sale of corn|Sale of corn from the demesne|Sale of corn from tithes| Sale of rye and oats|Sale of wheat|Sale of the mill's corn|wheat, rye, dredge, peas and oats sold", case=False, na=False, regex=True)]
            
            #check to verify
            print("Rows in sale_of_corn_prices_df:", sale_of_corn_prices_df.shape[0])
            print("Contents of 'Corn' in sale_of_corn_prices_df:", sale_of_corn_prices_df['Corn'])


            # Initial columns to include that are not dynamically named
            columns_to_include = [
                'Heading', 'Sentence', 'Total in Pounds', 'Pounds', 'Shillings', 'Pence', 'Corn',
                'Total Quarters Except 1', 'Total Bushels Except 1',
                'Total Corn Cross-Check', 'Total Corn Manorial Account', 'Corn Quantity Error'
            ]
        
            ####################### DYNAMICALLY ADD NEEDED COLUMNS #######################
            #Step1: Dynamically add 'Quarters x', 'Bushels x', 'shilling a quarter x', 'pence a quarter x'
            # Assuming the renaming has been correctly applied to sale_of_corn_prices_df
            for i in range(1, 31):  # Adjust the range as necessary, here it goes from 1 to 30
                columns_to_include.extend([
                    f'Quarters {i}', f'Bushels {i}',
                    f'shilling a quarter {i}', f'pence a quarter {i}'
                ])
            
            # Filter to include only columns that actually exist in raw_data_df
            columns_to_include = [col for col in columns_to_include if col in raw_data_df.columns]
            #Print to verify data is there
            print("Data in sale_of_corn_prices_df after filtering:")
            print(sale_of_corn_prices_df.head())
        
        
            # Now, use columns_to_include to filter the DataFrame
            sale_of_corn_prices_df = raw_data_df.loc[raw_data_df['Heading'].str.contains("Sale of Corn|Sale of corn|Sale of corn from the demesne|Sale of corn from tithes| Sale of rye and oats|Sale of wheat|Sale of the mill's corn|wheat, rye, dredge, peas and oats sold", case=False, na=False), columns_to_include]
            
            sale_of_corn_prices_df = sale_of_corn_prices_df[columns_to_include].copy()
            
            #Add the New DataFrame to the Dictionary of DataFrames
            all_sheets_df['Sale of Corn Prices'] = sale_of_corn_prices_df
            
            rename_dict = {}
            # Renaming "Quarters 1" and "Bushels 1" to represent totals as "Quarters 0" and "Bushels 0"
            rename_dict['Quarters 1'] = 'Quarters 0'
            rename_dict['Bushels 1'] = 'Bushels 0'
            
            # Apply the initial renaming to sale_of_corn_prices_df
            sale_of_corn_prices_df.rename(columns=rename_dict, inplace=True)
        
    
            #renaming rest of bushels and quarters
            rename_dict_rest = {}
            for i in range(2, 31):  
                rename_dict_rest[f'Quarters {i}'] = f'Quarters {i-1}'
                rename_dict_rest[f'Bushels {i}'] = f'Bushels {i-1}'
            
            # Apply the renaming for the rest to sale_of_corn_prices_df
            sale_of_corn_prices_df.rename(columns=rename_dict_rest, inplace=True)
            
            # Now, verify the renaming by printing the columns
            #print(sale_of_corn_prices_df.columns)
        
        
            # Assuming the renaming has been done as previously described
            for i in range(0, 29):  # Adjust the range according to the actual number of quarters you have
                # Construct the column names
                shilling_col = f'shilling a quarter {i}'
                pence_col = f'pence a quarter {i}'
                pound_col = f'£ a quarter {i}'
    
                # Check if the shilling and pence columns exist before attempting the calculation
                if shilling_col in sale_of_corn_prices_df.columns and pence_col in sale_of_corn_prices_df.columns:
                    # Calculate "£ a quarter"
                    sale_of_corn_prices_df[pound_col] = (
                        sale_of_corn_prices_df[shilling_col] / 20 +
                        sale_of_corn_prices_df[pence_col] / 240
                    )
            
            # Calculate the total quarters sold by adding 'Quarters 0' to 'Bushels 0' divided by 8
            sale_of_corn_prices_df['Quarters Sold'] = sale_of_corn_prices_df['Quarters 0'] + sale_of_corn_prices_df['Bushels 0'] / 8
            
            # Check if "£ a quarter 1" is empty (NaN) and calculate the price per quarter only for those rows
            condition = pd.isna(sale_of_corn_prices_df['£ a quarter 1'])
            
            # Apply the calculation where the condition is True
            sale_of_corn_prices_df.loc[condition, '£ a quarter 1'] = sale_of_corn_prices_df['Total in Pounds'] / sale_of_corn_prices_df['Quarters Sold']
            
            # Verify the conditional update
            #print(sale_of_corn_prices_df[['Total in Pounds', 'Quarters 0', 'Bushels 0', 'Quarters Sold', '£ a quarter 1']])
        
    
    
            ####################### CALCULATE PRICES #######################
            #Calculating weight and non-weighted average prices per quarter
            # Initialize columns for weighted sums and total quarters
            weighted_sum = 0
            total_quarters = 0
            prices = []
            
            # Loop through each set of '£ a quarter x', 'Quarters x', and 'Bushels x'
            for i in range(1, 31):  # Adjust the range as necessary, here it goes from 1 to 30
                price_col = f'£ a quarter {i}'
                quarters_col = f'Quarters {i}'
                bushels_col = f'Bushels {i}'
                
                # Check if the columns exist in the DataFrame
                if price_col in df.columns and quarters_col in df.columns and bushels_col in df.columns:
                    # Convert bushels to quarters and calculate total quarters for this price
                    quarters = df[quarters_col] + df[bushels_col] / 8
                    # Calculate weighted sum for this price
                    weighted_sum += df[price_col] * quarters
                    # Update total quarters sold
                    total_quarters += quarters
                    # Collect prices for non-weighted average calculation
                    prices.append(df[price_col])

            # Calculate weighted average price per quarter
            def calculate_weighted_average(row):
                weighted_sum = 0
                total_quarters = 0
                prices_count = 0  # Track the number of non-empty price columns
            
                for i in range(1, 31):  # Adjust range as needed
                    price_col = f'£ a quarter {i}'
                    quarters_col = f'Quarters {i}'
                    bushels_col = f'Bushels {i}'
                    if price_col in row.index and quarters_col in row.index and bushels_col in row.index and not pd.isna(row[price_col]):
                        quarters = row[quarters_col] + row[bushels_col] / 8
                        weighted_sum += row[price_col] * quarters
                        total_quarters += quarters
                        prices_count += 1
            
                # If only one price is available, return it directly
                if prices_count == 1:
                    return row['£ a quarter 1']
        
                # Otherwise, return the calculated weighted average
                return weighted_sum / total_quarters if total_quarters else None
        
            def calculate_non_weighted_average(row):
                prices = [row[f'£ a quarter {i}'] for i in range(1, 31) if f'£ a quarter {i}' in row and not pd.isna(row[f'£ a quarter {i}'])]
                return sum(prices) / len(prices) if prices else None
            
            # Apply the functions row-wise
            sale_of_corn_prices_df['Weighted Average Price per Quarter'] = sale_of_corn_prices_df.apply(calculate_weighted_average, axis=1)
            sale_of_corn_prices_df['Non-Weighted Average Price per Quarter'] = sale_of_corn_prices_df.apply(calculate_non_weighted_average, axis=1)
            
        
            ####################### MERGE SALE OF CORN PRICES INTO NEW SHEET BASED ON CORN TYPE #######################
            #Merge prices based on 'Corn' and 'Gross Output' 
            # Convert 'Gross Output' in new_sheet_df and 'Corn' in sale_of_corn_prices_df to lowercase for a case-insensitive merge
            new_sheet_df['Gross Output lowercase'] = new_sheet_df['Gross Output'].str.lower()
            sale_of_corn_prices_df['Corn lowercase'] = sale_of_corn_prices_df['Corn'].str.lower()
            
            # Remove duplicates based on 'Corn lowercase', keeping only the first occurrence
            #This gets rid of aftersales (supercompotum) - but check with original to make sure. 
            sale_of_corn_prices_df = sale_of_corn_prices_df.drop_duplicates(subset=['Corn lowercase'], keep='first')

            #check potentiall missing columns for easier debugging:
            required_columns = ['Corn lowercase', 'Weighted Average Price per Quarter', 'Non-Weighted Average Price per Quarter']
            missing_columns = [col for col in required_columns if col not in sale_of_corn_prices_df.columns]
            if missing_columns:
                print(f"Missing columns in sale_of_corn_prices_df: {missing_columns}")

            print("Keys in new_sheet_df before merge:", new_sheet_df['Gross Output lowercase'].unique())
            print("Keys in sale_of_corn_prices_df before merge:", sale_of_corn_prices_df['Corn lowercase'].unique())

            # Perform the merge using the lowercase columns
            merged_df = pd.merge(new_sheet_df, sale_of_corn_prices_df.dropna(subset=['Corn lowercase'])[['Corn lowercase', 'Weighted Average Price per Quarter', 'Non-Weighted Average Price per Quarter']],
                    left_on='Gross Output lowercase', right_on='Corn lowercase', how='left')

            print("Shape of merged_df after merge:", merged_df.shape)
            print("Head of merged_df after merge:", merged_df.head())
            if merged_df.empty:
                print(f"merged_df is empty after merging for {doc_name}, cannot proceed to save.")
            else:
            # Changed to pass so adderbury4 is still generated
                pass
        
            # Optionally, you can drop the temporary lowercase columns after the merge if they are no longer needed
            #merged_df.drop(columns=['Gross Output lowercase', 'Corn lowercase'], inplace=True)
            
            # Now, if you're observing duplicates in 'merged_df', it suggests there might be duplicates in 'new_sheet_df'
            # before merging or issues in how the merge keys uniquely identify rows.
            # Let's ensure there are no duplicates in 'new_sheet_df' before merging.
            new_sheet_df = new_sheet_df.drop_duplicates(subset=['Gross Output lowercase'], keep='first')

            #some print to identify bugs
            print("Before merge:")
            print(new_sheet_df.head())
            print(sale_of_corn_prices_df.dropna(subset=['Corn lowercase'])[['Corn lowercase', 'Weighted Average Price per Quarter', 'Non-Weighted Average Price per Quarter']].head())

                
            # Perform the merge again with updated 'new_sheet_df'
            merged_df = pd.merge(new_sheet_df, sale_of_corn_prices_df[['Corn lowercase', 'Weighted Average Price per Quarter', 'Non-Weighted Average Price per Quarter']],
                                    left_on='Gross Output lowercase', right_on='Corn lowercase', how='left')

            print("After merge:") 
            print(merged_df.head())
        
            # Drop the lowercase columns post-merge
            if not merged_df.empty: # Kuba: Changed to allow for creation of Adderbury
                merged_df.drop(columns=['Gross Output lowercase', 'Corn lowercase'], inplace=True)
                
                # Update the main dictionary with the cleaned, merged data
                all_sheets_df['Issues of the Grange'] = merged_df
            
            
                ####################### CALCULATE TOTAL VALUE OF PRODUCTION #######################
                # Calculate 'Value of Total Production weighted'
                merged_df['Value of Total Production weighted'] = merged_df['Account Total Quarters'] * merged_df['Weighted Average Price per Quarter']
                
                # Calculate 'Value of Total Production unweighted'
                merged_df['Value of Total Production unweighted'] = merged_df['Account Total Quarters'] * merged_df['Non-Weighted Average Price per Quarter']
                
            
                #This makes sure that the merged variables actually show up
                new_sheet_df = merged_df
                
                # Make sure to update 'Issues of the Grange' in all_sheets_df with the modified new_sheet_df
                all_sheets_df['Issues of the Grange'] = new_sheet_df
            else:
                print(f"No price data merged for {doc_name}, leaving 'Issues of the Grange' without value-of-production columns.")

    """ else:
        # If 'Issues of the Grange' is not present or 'Sentence' column is missing,
        # notify and skip processing related to it
        print("'Sentence' column not found in 'Issues of the Grange'. Skipping related processing.")
        

        ####################### FINAL SAVE #######################
        # Save all DataFrames to a new Excel file
        #output_file = "/Users/victoriagierok/Dropbox/My Mac (MacBook-Pro.fritz.box)/Desktop/Manorial1301_sorted2.xlsx"
        #output_file = '/Users/victoriagierok/Dropbox/Physical Capital England/Decennial 2018/Detailed Estimates/Winchester Pipe Rolls/1301-1302/python transcription/script4/Ecchinswell4.xlsx'

        print(f"Preparing to save file for {doc_name}.")
        print("Sheets available in all_sheets_df before saving:", all_sheets_df.keys())

                
        with pd.ExcelWriter(output_file) as writer:
            for sheet_name, df in all_sheets_df.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False, float_format="%.2f")
                        
        # Optionally add a print statement to confirm which file has been saved
        print(f"Saved processed data to {output_file}") """


    if 'Issues of the Grange' in all_sheets_df and 'Sentence' in all_sheets_df['Issues of the Grange'].columns:
        # Checks if manor succesfully printed before
        pass  
    else:
        # If 'Issues of the Grange' is not present or 'Sentence' column is missing,
        # notify and skip processing related to it
        print("'Sentence' column not found in 'Issues of the Grange'. Skipping related processing step.")
    
    ####################### FINAL SAVE (ALWAYS RUNS) #######################
    # Save all DataFrames to a new Excel file
    print(f"Preparing to save file for {doc_name}.")
    print("Sheets available in all_sheets_df before saving:", all_sheets_df.keys())

    with pd.ExcelWriter(output_file) as writer:
        for sheet_name, df in all_sheets_df.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False, float_format="%.2f")
                    
    # Optionally add a print statement to confirm which file has been saved
    print(f"Saved processed data to {output_file}")
        
    
     



Processing document: Adderbury
Loading Excel file: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script3\\Adderbury_1409_3.xlsx
Check1: Columns in issues_df after initialization: Index(['Heading', 'Sentence', 'Quarters 1', 'Bushels 1', 'Sown Quarters',
       'Sown Bushels', 'Sown Total Quarters', 'Bought in Quarters',
       'Bought in Bushels', 'Bought in Total Quarters', 'Tithe Quarters',
       'Tithe Bushels', 'Tithe Total Quarters',
       'Manorial servants & livestock Quarters',
       'Manorial servants & livestock Bushels',
       'Manorial servants & livestock Total Quarters', 'Sold Quarters',
       'Sold Bushels', 'Sold Total Quarters', 'Account Quarters',
       'Account Bushels', 'Account Total Quarters'],
      dtype='object')
Shape of issues_df before merge: (17, 23)
Shape of raw_data_df: (267, 30)
Shape of issues_df after merge: (17, 30)
Column 'Total Bought Quarters

C:\Users\kubak\AppData\Local\Temp\ipykernel_48096\1872698294.py:106: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  issues_df[f'{category} Quarters'].fillna(0, inplace=True)
C:\Users\kubak\AppData\Local\Temp\ipykernel_48096\1872698294.py:107: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

Saved processed data to C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script4\\Adderbury_1409_4.xlsx

Processing document: Alresford
Loading Excel file: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script3\\Alresford_1409_3.xlsx
Check1: Columns in issues_df after initialization: Index(['Heading', 'Sentence', 'Quarters 1', 'Bushels 1', 'Sown Quarters',
       'Sown Bushels', 'Sown Total Quarters', 'Bought in Quarters',
       'Bought in Bushels', 'Bought in Total Quarters', 'Tithe Quarters',
       'Tithe Bushels', 'Tithe Total Quarters',
       'Manorial servants & livestock Quarters',
       'Manorial servants & livestock Bushels',
       'Manorial servants & livestock Total Quarters', 'Sold Quarters',
       'Sold Bushels', 'Sold Total Quarters', 'Account Quarters',
       'Account 

C:\Users\kubak\AppData\Local\Temp\ipykernel_48096\1872698294.py:106: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  issues_df[f'{category} Quarters'].fillna(0, inplace=True)
C:\Users\kubak\AppData\Local\Temp\ipykernel_48096\1872698294.py:107: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

Saved processed data to C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script4\\Alresford_1409_4.xlsx

Processing document: Alverstoke
Loading Excel file: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script3\\Alverstoke_1409_3.xlsx
'Sentence' column not found in 'Issues of the Grange'. Skipping related processing step.
Preparing to save file for Alverstoke.
Sheets available in all_sheets_df before saving: dict_keys(['raw data', 'Receipts', 'Expenses', 'Overview', 'Issues of the Grange', 'Issues of the Mills', 'Stock', 'Prices', 'Labour Rents', 'not needed'])
Saved processed data to C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script4\\Alverstoke_1409_4.xlsx

Processing document: Ashma

...

Saved processed data to C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script4\\Wield_1409_4.xlsx

Processing document: Witney
Loading Excel file: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script3\\Witney_1409_3.xlsx
Check1: Columns in issues_df after initialization: Index(['Heading', 'Sentence', 'Quarters 1', 'Bushels 1', 'Sown Quarters',
       'Sown Bushels', 'Sown Total Quarters', 'Bought in Quarters',
       'Bought in Bushels', 'Bought in Total Quarters', 'Tithe Quarters',
       'Tithe Bushels', 'Tithe Total Quarters',
       'Manorial servants & livestock Quarters',
       'Manorial servants & livestock Bushels',
       'Manorial servants & livestock Total Quarters', 'Sold Quarters',
       'Sold Bushels', 'Sold Total Quarters', 'Account Quarters',
       'Account Bushels', 

C:\Users\kubak\AppData\Local\Temp\ipykernel_48096\1872698294.py:106: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  issues_df[f'{category} Quarters'].fillna(0, inplace=True)
C:\Users\kubak\AppData\Local\Temp\ipykernel_48096\1872698294.py:107: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

Saved processed data to C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script4\\Witney_1409_4.xlsx

Processing document: WitneyBorough
Loading Excel file: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script3\\WitneyBorough_1409_3.xlsx
'Sentence' column not found in 'Issues of the Grange'. Skipping related processing step.
Preparing to save file for WitneyBorough.
Sheets available in all_sheets_df before saving: dict_keys(['raw data', 'Receipts', 'Expenses', 'Overview', 'Issues of the Grange', 'Issues of the Mills', 'Stock', 'Prices', 'Labour Rents', 'not needed'])
Saved processed data to C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script4\\WitneyBorough_1409_4.xlsx

Processing docume

C:\Users\kubak\AppData\Local\Temp\ipykernel_48096\1872698294.py:106: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  issues_df[f'{category} Quarters'].fillna(0, inplace=True)
C:\Users\kubak\AppData\Local\Temp\ipykernel_48096\1872698294.py:107: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

After merge:
                                  Gross Output  Bought Only Aggregated  \
0  total of all allowances and cash deliveries                     0.0   
1                                        wheat                     0.0   
2                                       barley                     0.0   
3                                         oats                     0.0   
4                                      Mancorn                     0.0   

   Account Quarters  Account Bushels  Account Total Quarters  \
0                 0                0                     0.0   
1                 0                0                     0.0   
2                 0                0                     0.0   
3                 0                0                     0.0   
4                 0                0                     0.0   

   Bought in Quarters  Bought in Bushels  Total Bought Quarters  \
0                   0                  0                    0.0   
1                   0  

In [ ]:
####################### SCRIPT 5  ####################### 
import pandas as pd
import re
import numpy as np 


# This script populates the Stock sheet correctly


# Define document names and base paths >>> Kuba: Changed due to seemingly missing manors for this year. Ask V. 
import glob
import os

pattern = os.path.join(input_base_path, "*_1409_4.xlsx")
input_files = glob.glob(pattern)

document_names = [
    os.path.basename(f).replace("_1409_4.xlsx", "")
    for f in input_files
]

# Paths setup
stock_file_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\OCR 1409\stock.xlsx"
input_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script4\\"
output_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script5\\"

# Load the stock Excel sheet
#stock_file_path = "/Users/victoriagierok/Dropbox/My Mac (MacBook-Pro.fritz.box)/Desktop/stock.xlsx"
#stock_file_path = "/Users/victoriagierok/Dropbox/Physical Capital England/Decennial 2018/Detailed Estimates/Winchester Pipe Rolls/1301-1302/python transcription/stock.xlsx"

stock_df = pd.read_excel(stock_file_path, sheet_name='stock')

# Path to the original Excel file
#input_file_path = "/Users/victoriagierok/Dropbox/My Mac (MacBook-Pro.fritz.box)/Desktop/Manorial1301_sorted2.xlsx"
#input_file_path = '/Users/victoriagierok/Dropbox/Physical Capital England/Decennial 2018/Detailed Estimates/Winchester Pipe Rolls/1301-1302/python transcription/script4/Alverstoke4.xlsx'




##### Mapping all the types of Stock using Stock Excel
# Define the function to find and map "Stock" based on "Item" matches before a colon in "Sentence"
def find_stock(sentence):
    for _, row in stock_df.iterrows():
        item, stock_value = row['Item'], row['Stock']
        if re.search(rf'\b{re.escape(item)}\b(?=[:])', sentence, re.IGNORECASE):
            return stock_value
    return None


# Process each document
for doc_name in document_names:
    input_file_path = f"{input_base_path}{doc_name}_1409_4.xlsx"
    output_file_path = f"{output_base_path}{doc_name}_1409_5.xlsx"

    # Load all sheets into a dictionary of DataFrames
    all_sheets_df = pd.read_excel(input_file_path, sheet_name=None)


    if 'Stock' in all_sheets_df:
        stock_sheet_df = all_sheets_df['Stock']

    # Check if 'Stock' sheet exists and is not empty
    if 'Stock' in all_sheets_df and not all_sheets_df['Stock'].empty:
        stock_sheet_df = all_sheets_df['Stock']
        stock_sheet_df['stock'] = stock_sheet_df['Sentence'].apply(find_stock)
        stock_sheet_df['stock'] = stock_sheet_df['stock'].ffill()  # Forward fill 'stock' column
    
         # Drop unnecessary columns
        columns_to_delete = ["Total in Pounds", "Pounds", "Shillings", "Pence"]
        stock_sheet_df.drop(columns=columns_to_delete, inplace=True, errors='ignore')
    
        #### Extracting the numbers of various animals from the Sentence
        # Function to directly extract numbers for "added" and "murrain"
        def extract_number_before_keyword(sentence, keywords):
            for keyword in keywords:
            # Adjust the regex pattern to account for possible words between the number and the keyword
            # and to make sure the keyword is not followed by a full-stop or colon
                match = re.search(r'(\d+)\s+[^\.,;]*?\b' + re.escape(keyword) + r'\b', sentence)
                if match:
                    return match.group(1)
            return np.nan
    
        # Adjusted function for "inherited stock"
        def adjust_inherited_stock(row, found_matches):
            stock_type = row['stock']
            sentence = row['Sentence']
            # Only process if this stock type hasn't been matched yet
            if not found_matches.get(stock_type, False):
                match = re.search(r'(\d+)\s+[^\.,;]*?\bremain\b', sentence)
                if match:
                    found_matches[stock_type] = True  # Mark this stock type as matched
                    return match.group(1)
            return np.nan
    
        # Process other columns as necessary
        found_matches = {}  # Track matches for unique processing
        stock_sheet_df['inherited stock'] = stock_sheet_df.apply(lambda row: adjust_inherited_stock(row, found_matches), axis=1)
        stock_sheet_df['added'] = stock_sheet_df['Sentence'].apply(lambda x: extract_number_before_keyword(x, ["bought", "offspring"]))
        stock_sheet_df['murrain'] = stock_sheet_df['Sentence'].apply(lambda x: extract_number_before_keyword(x, ["in murrain"]))
        stock_sheet_df['sold'] = stock_sheet_df['Sentence'].apply(lambda x: extract_number_before_keyword(x, ["sold"]))
    
        # Update the dictionary with modified DataFrame
        all_sheets_df['Stock'] = stock_sheet_df
    else:
        print("The 'Stock' sheet is empty or does not exist. No processing will be applied to it.")
    
    

    # Save all DataFrames (sheets) to the new Excel file
    with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
        for sheet_name, df in all_sheets_df.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    print("All sheets, including the modified 'Stock' sheet, have been saved to the new file successfully.")


In [7]:
####################### SCRIPT 6.1 (NO TEMPLATE RECEIPTS/EXPENSES) #######################

import os
import glob
from pathlib import Path
from openpyxl import load_workbook

# ---------- PATHS ----------
input_base_path_5 = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script5\\"
template_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\archive\\"
template_filename = "Adderbury 1409_wl_format.xlsx"
output_base_path_6 = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script6\\"

Path(output_base_path_6).mkdir(parents=True, exist_ok=True)

template_path = os.path.join(template_base_path, template_filename)

# ---------- HELPERS ----------
def safe_remove_sheet(wb, sheet_name):
    """Remove sheet if present; no-op otherwise."""
    if sheet_name in wb.sheetnames:
        wb.remove(wb[sheet_name])

def find_sheet_by_keyword(wb, keyword):
    """Return the first sheet name containing keyword (case-insensitive), else None."""
    kw = keyword.lower()
    for name in wb.sheetnames:
        if kw in name.lower():
            return name
    return None

# ---------- PROCESS ----------
pattern = os.path.join(input_base_path_5, "*_1409_5.xlsx")
input_files = sorted(glob.glob(pattern))

print(f"Found {len(input_files)} script-5 workbooks")

for script5_file in input_files:
    filename_5 = os.path.basename(script5_file)          # e.g. 'Adderbury_1409_5.xlsx'
    doc_name = filename_5.replace("_1409_5.xlsx", "")    # e.g. 'Adderbury'

    # 1) Load template workbook
    wb = load_workbook(template_path)

    # 2) REMOVE template receipts/expenses sheets so they are NOT copied into outputs
    # Remove exact Adderbury-named sheets (common in your template)
    safe_remove_sheet(wb, "Adderbury Receipts")
    safe_remove_sheet(wb, "Adderbury Expenses")

    # Also remove any remaining receipts/expenses skeleton sheets by keyword (defensive)
    sh_receipts = find_sheet_by_keyword(wb, "receipts")
    sh_expenses = find_sheet_by_keyword(wb, "expenses")
    if sh_receipts:
        safe_remove_sheet(wb, sh_receipts)
    if sh_expenses:
        safe_remove_sheet(wb, sh_expenses)

    # 3) Rename Stock sheet only (optional; keep if your downstream expects manor-named stock)
    try:
        ws_stock = wb["Adderbury Stock"]
        ws_stock.title = f"{doc_name} Stock"
        ws_stock["A1"] = f"{doc_name.upper()} STOCK ACCOUNT"
    except KeyError:
        # If the template has no stock sheet under that exact name, try keyword match
        stock_name = find_sheet_by_keyword(wb, "stock")
        if stock_name:
            ws_stock = wb[stock_name]
            ws_stock.title = f"{doc_name} Stock"
            ws_stock["A1"] = f"{doc_name.upper()} STOCK ACCOUNT"
        else:
            print(f"{doc_name}: template has no stock sheet to rename.")

    # 4) Load script-5 workbook and copy its sheets across (values only)
    wb5 = load_workbook(script5_file, data_only=True)

    for src_ws in wb5.worksheets:
        src_title = src_ws.title

        # If a sheet with this name already exists in template, remove it so we replace it
        if src_title in wb.sheetnames:
            wb.remove(wb[src_title])

        dest_ws = wb.create_sheet(src_title)

        for row in src_ws.iter_rows():
            for cell in row:
                dest_ws[cell.coordinate].value = cell.value

    # 5) Save script-6 output
    output_file = os.path.join(output_base_path_6, filename_5.replace("_5.xlsx", "_6.xlsx"))
    wb.save(output_file)
    print(f"Created script-6 workbook: {output_file}")


Found 56 script-5 workbooks
Created script-6 workbook: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script6\\Adderbury_1409_6.xlsx
Created script-6 workbook: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script6\\Alresford_1409_6.xlsx
Created script-6 workbook: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script6\\Alverstoke_1409_6.xlsx
Created script-6 workbook: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script6\\Ashmansworth_1409_6.xlsx
Created script-6 workbook: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 

In [8]:
####################### SCRIPT 6.2 - Expenses (REORDER COLUMNS + FORMAT + CLEAN + SUMMARISE) #######################
import os
import glob
from openpyxl import load_workbook
from openpyxl.styles import Border, Side, PatternFill
from copy import copy as copy_style

# Folder with script-6 outputs
script6_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script6\\"

# Template workbook with correct formatting (Adderbury)
template_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\archive\Adderbury 1409_wl_format.xlsx"

template_wb = load_workbook(template_path)
template_ws = template_wb["Adderbury Expenses"]
print("Template sheet loaded:", template_ws.title)


# -------------------- HELPERS --------------------
def find_header_row(ws, header_text="Heading", max_check_rows=40):
    """
    Find the row containing the header cell 'Heading' (exact match).
    Searches the first max_check_rows rows across all columns.
    """
    max_check_rows = min(ws.max_row, max_check_rows)
    for r in range(1, max_check_rows + 1):
        for c in range(1, ws.max_column + 1):
            v = ws.cell(row=r, column=c).value
            if isinstance(v, str) and v.strip() == header_text:
                return r
    return None

def reorder_expenses_columns(wb, ws_old, manor_name):
    """
    Rebuild 'Expenses' into:
    A Heading | B £ | C s | D d | E (blank) | F Total in £ | G Error | H Notes

    Column sources are detected from the header row by header text.
    A1 is set to '{MANOR} EXPENSES' and will not be overwritten.
    """

    # --- detect header row (row containing 'Heading') ---
    hdr_old = find_header_row(ws_old, "Heading", max_check_rows=60)
    if hdr_old is None:
        hdr_old = 2  # fallback
    hdr_new = max(2, hdr_old)  # IMPORTANT: never allow header row to be 1

    # --- detect source columns by header labels on hdr_old ---
    def normh(x):
        return str(x).strip().lower() if x is not None else ""

    header_map = {}
    for c in range(1, ws_old.max_column + 1):
        header_map[normh(ws_old.cell(row=hdr_old, column=c).value)] = c

    # Try a few variants that appear in your files
    col_heading = header_map.get("heading", 1)
    col_total   = header_map.get("total in £") or header_map.get("total in pounds") or header_map.get("total in p") or 3
    col_pounds  = header_map.get("pounds") or header_map.get("£") or 4
    col_shill   = header_map.get("shillings") or header_map.get("s") or 6
    col_pence   = header_map.get("pence") or header_map.get("d") or 7
    col_error   = header_map.get("error") or 9   # <- this will pick I if Error is there
    col_notes   = header_map.get("sentence")        # may not exist

    # --- create temp sheet and copy values into new layout ---
    tmp_name = "Expenses__tmp"
    if tmp_name in wb.sheetnames:
        wb.remove(wb[tmp_name])
    ws_new = wb.create_sheet(tmp_name)

    max_row = ws_old.max_row

    for r in range(1, max_row + 1):
        # A: Heading
        ws_new.cell(row=r, column=1).value = ws_old.cell(row=r, column=col_heading).value

        # B: £  (integer pounds)
        ws_new.cell(row=r, column=2).value = ws_old.cell(row=r, column=col_pounds).value

        # C: s
        ws_new.cell(row=r, column=3).value = ws_old.cell(row=r, column=col_shill).value

        # D: d
        ws_new.cell(row=r, column=4).value = ws_old.cell(row=r, column=col_pence).value

        # E: blank spacer
        ws_new.cell(row=r, column=5).value = None

        # F: Total in £
        ws_new.cell(row=r, column=6).value = ws_old.cell(row=r, column=col_total).value

        # G: Error
        ws_new.cell(row=r, column=7).value = ws_old.cell(row=r, column=col_error).value if col_error else None

        # H: Notes
        ws_new.cell(row=r, column=8).value = ws_old.cell(row=r, column=col_notes).value if col_notes else None

        # I: Fixed Investments (blank)
        ws_new.cell(row=r, column=9).value = None

        # J: Demesne Fixed Investments (blank)
        ws_new.cell(row=r, column=10).value = None


    # --- write header labels on hdr_new (not row 1) ---
    ws_new.cell(row=hdr_new, column=1).value = "Heading"
    ws_new.cell(row=hdr_new, column=2).value = "£"
    ws_new.cell(row=hdr_new, column=3).value = "s"
    ws_new.cell(row=hdr_new, column=4).value = "d"
    ws_new.cell(row=hdr_new, column=5).value = None
    ws_new.cell(row=hdr_new, column=6).value = "Total in £"
    ws_new.cell(row=hdr_new, column=7).value = "Error"
    ws_new.cell(row=hdr_new, column=8).value = "Sentence"
    ws_new.cell(row=2, column=9).value = "Fixed Investments"
    ws_new.cell(row=2, column=10).value = "Demesne Fixed Investments"
    
    # --- enforce title row 1 after headers so it cannot be overwritten ---
    ws_new.cell(row=1, column=1).value = f"{manor_name.upper()} EXPENSES"
    for c in range(2, 9):
        ws_new.cell(row=1, column=c).value = None

    return ws_new

def apply_expenses_formatting(target_ws, template_ws):
    """
    Copy formatting from template_ws to target_ws WITHOUT touching values.

    Adjusted for columns A..H (not A..K).
    """
    # 1) Column widths
    for col_letter, col_dim in template_ws.column_dimensions.items():
        if col_dim.width is not None:
            # Only apply widths for A..H
            if col_letter in list("ABCDEFGH"):
                target_ws.column_dimensions[col_letter].width = col_dim.width

    # 2) Row heights
    for row_idx, row_dim in template_ws.row_dimensions.items():
        if row_dim.height is not None:
            target_ws.row_dimensions[row_idx].height = row_dim.height

    # 3) Determine header row (row containing 'Heading')
    header_row = find_header_row(target_ws, "Heading", max_check_rows=40)
    if header_row is None:
        header_row = 10

    # 4) Copy cell styles only for rows 1..header_row and columns A..H
    col_letters = list("ABCDEFGH")
    for row in range(1, header_row + 1):
        for col_letter in col_letters:
            tmpl_cell = template_ws[f"{col_letter}{row}"]
            tgt_cell = target_ws[f"{col_letter}{row}"]
            tgt_cell._style = copy_style(tmpl_cell._style)

    # 5) Copy merged cells exactly as in template (but clear existing first)
    target_ws.merged_cells.ranges = []
    for merged_range in template_ws.merged_cells.ranges:
        target_ws.merge_cells(str(merged_range))

    # 6) Borders
    thin = Side(style="thin", color="000000")
    max_row = target_ws.max_row
    col_letters = list("ABCDEFGH")

    # Clear borders in A..H
    for row in range(1, max_row + 1):
        for col_letter in col_letters:
            target_ws[f"{col_letter}{row}"].border = Border()

    # Right border on column A for all rows
    for row in range(1, max_row + 1):
        target_ws[f"A{row}"].border = Border(right=thin)

    # Bottom border on header row across A..H
    for col_letter in col_letters:
        cell = target_ws[f"{col_letter}{header_row}"]
        if col_letter == "A":
            cell.border = Border(right=thin, bottom=thin)
        else:
            cell.border = Border(bottom=thin)


def clean_and_style_expenses_data(ws):
    """
    Updated for new layout:
      - Column F (6) is Total in £
      - Columns A..H used
      - Category totals bolded etc.
    """
    col_letters = list("ABCDEFGH")

    # 1) Remove 'Total of all receipts' with 0/None in F
    for r in range(ws.max_row, 1, -1):
        a_val = ws.cell(row=r, column=1).value
        f_val = ws.cell(row=r, column=6).value  # Total in £
        if isinstance(a_val, str) and a_val.strip().lower() == "total of all receipts":
            is_zero_or_none = False
            if f_val is None:
                is_zero_or_none = True
            else:
                try:
                    is_zero_or_none = float(str(f_val).replace(",", ".")) == 0.0
                except ValueError:
                    is_zero_or_none = False
            if is_zero_or_none:
                ws.delete_rows(r, 1)

    # 2) Compress sequences of empty rows to a single row
    prev_empty = False
    for r in range(ws.max_row, 1, -1):
        row_vals = [ws.cell(row=r, column=c).value for c in range(1, len(col_letters) + 1)]
        is_empty = all(v in (None, "") for v in row_vals)
        if is_empty and prev_empty:
            ws.delete_rows(r, 1)
            continue
        prev_empty = is_empty

    # 3) Category block styling based on column A labels
    summary_exclusions = {
        "total of all expenses",
        "total of all expenses per account",
        "total of all expenses (calculated here)",
        "total fixed investment",
        "total demesne fixed investment",
    }

    categories = {}
    for r in range(1, ws.max_row + 1):
        val = ws.cell(row=r, column=1).value
        if not isinstance(val, str):
            continue
        label = val.strip()
        if not label:
            continue
        lower = label.lower()
        if lower == "heading":
            continue
        if lower.endswith(" total"):
            continue
        if lower == "total of all receipts":
            continue
        if lower in summary_exclusions:
            continue
        categories.setdefault(lower, {"label": label, "rows": []})["rows"].append(r)

    grey_fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")

    for cat in categories.values():
        label = cat["label"]
        rows_for_cat = cat["rows"]
        if not rows_for_cat:
            continue
        first_row = rows_for_cat[0]
        last_row = rows_for_cat[-1]

        for r in rows_for_cat:
            cell_a = ws[f"A{r}"]
            if len(rows_for_cat) == 1:
                cell_a.font = cell_a.font.copy(bold=True)
            else:
                if r == last_row:
                    cell_a.value = f"{label} total"
                    for col_letter in list("ABCDEFGH"):
                        cell = ws[f"{col_letter}{r}"]
                        cell.font = cell.font.copy(bold=True)
                        cell.fill = grey_fill
                elif r == first_row:
                    cell_a.font = cell_a.font.copy(bold=True)


def add_expenses_summary_rows(ws):
    """
    Updated for new layout:
      - Totals live in column F (6).
      - Sheet width A..H.
      - Copy B,C,D from 'Total of all expenses' row (these are £, s, d).
    """
    summary_labels = [
        "Total of all expenses per account",
        "Total of all expenses (calculated here)",
        "Total fixed investment",
        "Total Demesne fixed investment",
    ]
    summary_labels_lower = {lbl.lower() for lbl in summary_labels}

    # Remove existing summary rows
    for r in range(ws.max_row, 1, -1):
        val = ws.cell(row=r, column=1).value
        if isinstance(val, str) and val.strip().lower() in summary_labels_lower:
            ws.delete_rows(r, 1)

    # Find row containing "Total of all expenses"
    total_expenses_row = None
    for r in range(1, ws.max_row + 1):
        val = ws.cell(row=r, column=1).value
        if isinstance(val, str) and val.strip().lower() == "total of all expenses":
            total_expenses_row = r
            break
    if total_expenses_row is None:
        return

    # Remove trailing empty rows
    while True:
        last_row = ws.max_row
        row_vals = [ws.cell(row=last_row, column=c).value for c in range(1, 9)]
        if all(v in (None, "") for v in row_vals):
            ws.delete_rows(last_row, 1)
        else:
            break

    # Insert exactly one empty row before summaries
    blank_row = ws.max_row + 1
    ws.insert_rows(blank_row, 1)
    start_row = blank_row + 1

    def bold(cell):
        cell.font = cell.font.copy(bold=True)

    # Row 1: Total of all expenses per account
    ws.cell(row=start_row, column=1).value = "Total of all expenses per account"
    bold(ws.cell(row=start_row, column=1))

    # Copy £ s d from Total of all expenses row (B,C,D)
    for col in (2, 3, 4):
        ws.cell(row=start_row, column=col).value = ws.cell(row=total_expenses_row, column=col).value

    # F references Total in £
    ws.cell(row=start_row, column=6).value = f"=F{total_expenses_row}"

    # Row 2: calculated here
    ws.cell(row=start_row + 1, column=1).value = "Total of all expenses (calculated here)"
    bold(ws.cell(row=start_row + 1, column=1))
    ws.cell(row=start_row + 1, column=6).value = (
        f'=SUMIF(A1:A{total_expenses_row-1},"* total",F1:F{total_expenses_row-1})'
    )

    # Row 3: fixed investment
    ws.cell(row=start_row + 2, column=1).value = "Total fixed investment"
    bold(ws.cell(row=start_row + 2, column=1))

    # Row 4: demesne fixed investment
    ws.cell(row=start_row + 3, column=1).value = "Total Demesne fixed investment"
    bold(ws.cell(row=start_row + 3, column=1))


# -------------------- MAIN --------------------
pattern = os.path.join(script6_base_path, "*_1409_6.xlsx")
files = glob.glob(pattern)
print(f"Found {len(files)} script-6 workbooks")

for file_path in files:
    filename = os.path.basename(file_path)
    manor_name = filename.replace("_1409_6.xlsx", "")

    try:
        wb = load_workbook(file_path)

        if "Expenses" not in wb.sheetnames:
            print(f"{filename}: no 'Expenses' sheet found, skipping.")
            wb.close()
            continue

        ws_old = wb["Expenses"]

        # 0) Reorder columns by rebuilding the sheet
        ws_new = reorder_expenses_columns(wb, ws_old, manor_name)

        # Delete old sheet and rename new to "Expenses"
        wb.remove(ws_old)
        ws_new.title = "Expenses"

        # 1) Apply header formatting from template (A..H)
        apply_expenses_formatting(ws_new, template_ws)

        # 2) Clean + style categories/totals
        clean_and_style_expenses_data(ws_new)

        # 3) Add summary rows at end
        add_expenses_summary_rows(ws_new)

        wb.save(file_path)
        wb.close()
        print(f"{filename}: Expenses columns reordered + formatted + summarised.")

    except Exception as e:
        print(f"Error processing {filename}: {e}")


Template sheet loaded: Adderbury Expenses
Found 56 script-6 workbooks


C:\Users\kubak\AppData\Local\Temp\ipykernel_36088\1022807393.py:259: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell_a.font = cell_a.font.copy(bold=True)
C:\Users\kubak\AppData\Local\Temp\ipykernel_36088\1022807393.py:268: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell_a.font = cell_a.font.copy(bold=True)
C:\Users\kubak\AppData\Local\Temp\ipykernel_36088\1022807393.py:265: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.font = cell.font.copy(bold=True)
C:\Users\kubak\AppData\Local\Temp\ipykernel_36088\1022807393.py:317: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.font = cell.font.copy(bold=True)


Adderbury_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
Alresford_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
Alverstoke_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
Ashmansworth_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
Beauworth_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
Bentley_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
Bereleigh_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
BishopsFonthill_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
BishopsSutton_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
Bishopstone_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
BishopsWaltham_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
Bitterne_1409_6.xlsx: Expenses columns reordered + formatted + summarised.
Brightwell_1409_6.xlsx: Expenses columns reordered + formatted + summa

In [9]:
####################### SCRIPT 6.3 - Receipts  ####################### 

import os
import glob
from pathlib import Path
from copy import copy as copy_style

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Border, Side

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
script6_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script6\\"
template_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\archive\Adderbury 1409_wl_format.xlsx"

template_wb = load_workbook(template_path)
template_ws_receipts = template_wb["Adderbury Receipts"]   # adjust if sheet name differs


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def find_header_row(ws, header_label="Heading", max_search_rows=30):
    """Find the row index that contains the given header label."""
    for r in range(1, max_search_rows + 1):
        for c in range(1, ws.max_column + 1):
            val = ws.cell(row=r, column=c).value
            if isinstance(val, str) and val.strip() == header_label:
                return r
    raise ValueError(f"Could not find header '{header_label}' in sheet '{ws.title}'")


def reshape_receipts_sheet(ws, manor_name: str):
    """
    Reorder Receipts sheet columns to:

      A: Heading
      B: £
      C: s
      D: d
      E: (blank)
      F: Total in £
      G: Error
      H: Notes
      I: Own notes
    """

    # 1) Find existing header row before touching A1
    header_row = find_header_row(ws, "Heading")

    # 2) If header is at row 1, insert a new top row for the title
    if header_row == 1:
        ws.insert_rows(1)
        header_row += 1

    # 3) Title in A1
    ws["A1"] = f"{manor_name.upper()} RECEIPTS"

    # 4) Map original headers
    header_cells = ws[header_row]
    header_map = {}
    for cell in header_cells:
        if cell.value is None:
            continue
        label = str(cell.value).strip()
        header_map[label] = cell.column

    required_old = [
        "Heading",
        "Sentence",
        "Total in Pounds",
        "Pounds",
        "Shillings",
        "Pence",
        "Error",
    ]
    missing = [h for h in required_old if h not in header_map]
    if missing:
        raise ValueError(f"Receipts sheet '{ws.title}' missing columns: {missing}")

    # 5) Clear the header row completely
    for c in range(1, ws.max_column + 1):
        ws.cell(row=header_row, column=c).value = None

    # 6) New header layout
    new_headers = [
        "Heading",
        "£",
        "s",
        "d",
        "",             # blank E
        "Total in £",
        "Error",
        "Notes",
        "Own notes",
    ]

    for col_idx, label in enumerate(new_headers, start=1):
        ws.cell(row=header_row, column=col_idx, value=label if label else None)

    # 7) Re-map data rows
    max_row = ws.max_row
    for row in range(header_row + 1, max_row + 1):
        row_vals = {
            name: ws.cell(row=row, column=col_idx).value
            for name, col_idx in header_map.items()
        }

        ws.cell(row=row, column=1).value = row_vals.get("Heading")
        ws.cell(row=row, column=2).value = row_vals.get("Pounds")
        ws.cell(row=row, column=3).value = row_vals.get("Shillings")
        ws.cell(row=row, column=4).value = row_vals.get("Pence")
        ws.cell(row=row, column=5).value = None                      # blank
        ws.cell(row=row, column=6).value = row_vals.get("Total in Pounds")
        ws.cell(row=row, column=7).value = row_vals.get("Error")
        ws.cell(row=row, column=8).value = row_vals.get("Sentence")  # Notes
        ws.cell(row=row, column=9).value = None                      # Own notes

    # 8) Remove any columns beyond I
    if ws.max_column > 9:
        ws.delete_cols(10, ws.max_column - 9)


def apply_receipts_header_formatting(target_ws, template_ws):
    """
    Apply header-style formatting from the Adderbury Receipts template:

      - Column widths, row heights
      - Styles in rows 1..header_row for columns A..I
      - Borders:
          * Right border on column A for all rows
          * Bottom border on header_row across A..I
    """

    # Column widths
    for col_letter, col_dim in template_ws.column_dimensions.items():
        if col_dim.width is not None:
            target_ws.column_dimensions[col_letter].width = col_dim.width

    # Row heights
    for row_idx, row_dim in template_ws.row_dimensions.items():
        if row_dim.height is not None:
            target_ws.row_dimensions[row_idx].height = row_dim.height

    # Header row in template
    header_row = find_header_row(template_ws, "Heading")
    col_letters = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]

    # Styles for header area
    for row in range(1, header_row + 1):
        for col_letter in col_letters:
            tmpl_cell = template_ws[f"{col_letter}{row}"]
            tgt_cell = target_ws[f"{col_letter}{row}"]
            tgt_cell._style = copy_style(tmpl_cell._style)

    # Merged cells from template
    target_ws.merged_cells.ranges = []
    for merged_range in template_ws.merged_cells.ranges:
        target_ws.merge_cells(str(merged_range))

    # Borders
    thin = Side(style="thin", color="000000")
    max_row = target_ws.max_row

    # Clear borders
    for row in range(1, max_row + 1):
        for col_letter in col_letters:
            target_ws[f"{col_letter}{row}"].border = Border()

    # Right border on A
    for row in range(1, max_row + 1):
        target_ws[f"A{row}"].border = Border(right=thin)

    # Bottom border on header row across A..I
    for col_letter in col_letters:
        cell = target_ws[f"{col_letter}{header_row}"]
        if col_letter == "A":
            cell.border = Border(right=thin, bottom=thin)
        else:
            cell.border = Border(bottom=thin)


def compress_empty_rows(ws):
    """
    Collapse any sequence of more than one empty row (below header)
    into a single empty row.
    """
    header_row = find_header_row(ws, "Heading")
    max_row = ws.max_row
    col_count = 9  # A..I

    prev_empty = False
    for r in range(max_row, header_row + 1, -1):  # bottom-up
        row_vals = [ws.cell(row=r, column=c).value for c in range(1, col_count + 1)]
        is_empty = all(v in (None, "") for v in row_vals)

        if is_empty and prev_empty:
            ws.delete_rows(r, 1)
            continue

        prev_empty = is_empty


def style_receipts_categories(ws):
    """
    Category styling for Receipts:

      - Group by unique label in column A (below header).
      - Skip headers, summary labels and labels ending with ' total'.
      - First occurrence: bold A.
      - Last occurrence: rename A to '<label> total' and make entire row A..I
        bold with grey fill.
      - Fix numeric totals for categories where the last-row F is 0/empty by
        summing F of earlier rows in that category (and setting G = 0).
    """

    header_row = find_header_row(ws, "Heading")
    max_row = ws.max_row

    exclusions = {
        "heading",
        "total of all receipts",
        "total of all expenses",
        "total of all receipts recorded in accounts",
        "total of all receipts recorded in accounts (calculated here)",
        "total of all receipts (calculated here) incl arrears",
    }

    categories = {}  # lower label -> {'label': original, 'rows': [...]}

    for r in range(header_row + 1, max_row + 1):
        val = ws.cell(row=r, column=1).value
        if not isinstance(val, str):
            continue

        label = val.strip()
        if not label:
            continue

        lower = label.lower()
        if lower in exclusions:
            continue
        if lower.endswith(" total"):
            continue

        key = lower
        if key not in categories:
            categories[key] = {"label": label, "rows": []}
        categories[key]["rows"].append(r)

    grey_fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2",
                            fill_type="solid")
    col_letters = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]

    def parse_number(val):
        if val is None:
            return None
        s = str(val).replace(",", ".")
        try:
            return float(s)
        except ValueError:
            return None

    for cat in categories.values():
        label = cat["label"]
        rows_for_cat = cat["rows"]
        if not rows_for_cat:
            continue

        first_row = rows_for_cat[0]
        last_row = rows_for_cat[-1]

        # Fix numeric total if last row F is 0/empty
        f_last_val = ws.cell(row=last_row, column=6).value
        f_last_num = parse_number(f_last_val)
        if f_last_num is None or abs(f_last_num) < 1e-9:
            total = 0.0
            has_any = False
            for r in rows_for_cat[:-1]:
                v = parse_number(ws.cell(row=r, column=6).value)
                if v is not None:
                    total += v
                    has_any = True
            if has_any:
                ws.cell(row=last_row, column=6).value = total
                ws.cell(row=last_row, column=7).value = 0  # Error = 0

        # Formatting: first/last rows
        for r in rows_for_cat:
            cell_a = ws[f"A{r}"]
            if len(rows_for_cat) == 1:
                cell_a.font = cell_a.font.copy(bold=True)
            else:
                if r == last_row:
                    cell_a.value = f"{label} total"
                    for col_letter in col_letters:
                        cell = ws[f"{col_letter}{r}"]
                        cell.font = cell.font.copy(bold=True)
                        cell.fill = grey_fill
                elif r == first_row:
                    cell_a.font = cell_a.font.copy(bold=True)
                else:
                    pass

def add_receipts_summary_rows(ws):
    """
    Add the final two summary rows at the end of the Receipts sheet:

      Row 1: 'Total of all receipts recorded in accounts (calculated here)'
              F = SUM of all '* total' rows above, minus 'Arrears total'.
      Row 2: 'Total of all receipts (calculated here) incl arrears'
              F = Row1.F + 'Arrears total' (if present).
              G = F('Total of all receipts') - F(Row 2).

    Existing rows with these labels are removed and recreated.
    """

    label1 = "Total of all receipts recorded in accounts (calculated here)"
    label2 = "Total of all receipts (calculated here) incl arrears"
    labels_lower = {label1.lower(), label2.lower()}

    # Remove any existing such summary rows
    for r in range(ws.max_row, 1, -1):
        val = ws.cell(row=r, column=1).value
        if isinstance(val, str) and val.strip().lower() in labels_lower:
            ws.delete_rows(r, 1)

    # Find the main "Total of all receipts" row (for the error calculation)
    total_receipts_row = None
    for r in range(1, ws.max_row + 1):
        val = ws.cell(row=r, column=1).value
        if isinstance(val, str) and val.strip().lower() == "total of all receipts":
            total_receipts_row = r
            break

    last_data_row = ws.max_row  # last row before new summaries

    def bold(cell):
        cell.font = cell.font.copy(bold=True)

    # ---- Row 1: Total of all receipts recorded in accounts (calculated here)
    row1 = last_data_row + 1
    ws.cell(row=row1, column=1).value = label1
    bold(ws.cell(row=row1, column=1))

    # Sum of all '* total' rows excluding 'Arrears total'
    formula1 = (
        f'=SUMIF(A1:A{last_data_row},"* total",F1:F{last_data_row})'
        f'-SUMIF(A1:A{last_data_row},"Arrears total",F1:F{last_data_row})'
    )
    ws.cell(row=row1, column=6).value = formula1

    # ---- Row 2: Total of all receipts (calculated here) incl arrears
    row2 = row1 + 1
    ws.cell(row=row2, column=1).value = label2
    bold(ws.cell(row=row2, column=1))

    # F = Row1.F + Arrears total (if present)
    formula2 = (
        f'=F{row1}'
        f'+SUMIF(A1:A{last_data_row},"Arrears total",F1:F{last_data_row})'
    )
    ws.cell(row=row2, column=6).value = formula2

    # ---- G (Error) = Total of all receipts (F) - this calculated incl arrears
    if total_receipts_row is not None:
        error_formula = f"=F{total_receipts_row}-F{row2}"
        ws.cell(row=row2, column=7).value = error_formula


# ------------------------------------------------------------
# Main loop over script-6 workbooks (Receipts / script 6.3)
# ------------------------------------------------------------
pattern = os.path.join(script6_base_path, "*_1409_6.xlsx")
files = glob.glob(pattern)
print(f"Found {len(files)} workbooks")

for file_path in files:
    filename = os.path.basename(file_path)
    manor_name = filename.replace("_1409_6.xlsx", "")

    try:
        wb = load_workbook(file_path)

        if "Receipts" not in wb.sheetnames:
            print(f"{filename}: no 'Receipts' sheet, skipping.")
            continue

        ws_receipts = wb["Receipts"]

        # 1) Reorder and title
        reshape_receipts_sheet(ws_receipts, manor_name)

        # 2) Header formatting from Adderbury Receipts template
        apply_receipts_header_formatting(ws_receipts, template_ws_receipts)

        # 3) Compress empty rows
        compress_empty_rows(ws_receipts)

        # 4) Category styling and numeric fixes
        style_receipts_categories(ws_receipts)

        # 5) Summary rows
        add_receipts_summary_rows(ws_receipts)

        wb.save(file_path)
        print(f"{filename}: Receipts processed (script 6.3).")

    except Exception as e:
        print(f"Error processing {filename}: {e}")


Found 56 workbooks


C:\Users\kubak\AppData\Local\Temp\ipykernel_36088\458106799.py:302: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell_a.font = cell_a.font.copy(bold=True)
C:\Users\kubak\AppData\Local\Temp\ipykernel_36088\458106799.py:299: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.font = cell.font.copy(bold=True)
C:\Users\kubak\AppData\Local\Temp\ipykernel_36088\458106799.py:340: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.font = cell.font.copy(bold=True)


Adderbury_1409_6.xlsx: Receipts processed (script 6.3).
Alresford_1409_6.xlsx: Receipts processed (script 6.3).
Alverstoke_1409_6.xlsx: Receipts processed (script 6.3).
Ashmansworth_1409_6.xlsx: Receipts processed (script 6.3).
Beauworth_1409_6.xlsx: Receipts processed (script 6.3).
Bentley_1409_6.xlsx: Receipts processed (script 6.3).
Bereleigh_1409_6.xlsx: Receipts processed (script 6.3).
BishopsFonthill_1409_6.xlsx: Receipts processed (script 6.3).
BishopsSutton_1409_6.xlsx: Receipts processed (script 6.3).
Bishopstone_1409_6.xlsx: Receipts processed (script 6.3).
BishopsWaltham_1409_6.xlsx: Receipts processed (script 6.3).
Bitterne_1409_6.xlsx: Receipts processed (script 6.3).
Brightwell_1409_6.xlsx: Receipts processed (script 6.3).
Burghclere_1409_6.xlsx: Receipts processed (script 6.3).
Cheriton_1409_6.xlsx: Receipts processed (script 6.3).
Crawley_1409_6.xlsx: Receipts processed (script 6.3).
Culham_1409_6.xlsx: Receipts processed (script 6.3).
DowntonBorough_1409_6.xlsx: Receip

C:\Users\kubak\AppData\Local\Temp\ipykernel_36088\458106799.py:293: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell_a.font = cell_a.font.copy(bold=True)


Holway_1409_6.xlsx: Receipts processed (script 6.3).
Ivinghoe_1409_6.xlsx: Receipts processed (script 6.3).
Merdon_1409_6.xlsx: Receipts processed (script 6.3).
Morton_1409_6.xlsx: Receipts processed (script 6.3).
Newtown_1409_6.xlsx: Receipts processed (script 6.3).
NorthWaltham_1409_6.xlsx: Receipts processed (script 6.3).
Otterford_1409_6.xlsx: Receipts processed (script 6.3).
OvertonBorough_1409_6.xlsx: Receipts processed (script 6.3).
Poundisford_1409_6.xlsx: Receipts processed (script 6.3).
Rimpton_1409_6.xlsx: Receipts processed (script 6.3).
Southwark_1409_6.xlsx: Receipts processed (script 6.3).
Staplegrove_1409_6.xlsx: Receipts processed (script 6.3).
StGilesFair_1409_6.xlsx: Receipts processed (script 6.3).
TauntonBorough_1409_6.xlsx: Receipts processed (script 6.3).
Taunton_1409_6.xlsx: Receipts processed (script 6.3).
Twyford_1409_6.xlsx: Receipts processed (script 6.3).
Upton_1409_6.xlsx: Receipts processed (script 6.3).
WalthamStLawrence_1409_6.xlsx: Receipts processed (

In [10]:
####################### SCRIPT 6.4 – STOCK (WORKING BLOCK) #######################

import os
import glob
import re
from openpyxl import load_workbook
import re

script6_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script6\\"

pattern = os.path.join(script6_base_path, "*_1409_6.xlsx")
input_files = sorted(glob.glob(pattern))

print(f"Found {len(input_files)} workbooks")


# ---------------- HELPERS ----------------
def norm(x):
    if x is None:
        return ""
    s = str(x).replace("\u00A0", " ").strip().lower()
    s = " ".join(s.split())
    s = s.replace("–", "-").replace("—", "-")
    s = s.rstrip(":;.,")
    return s

def key_id(x):
    """
    Matching key for cross-sheet joins.
    Removes spaces + hyphens so 'bare skins' == 'bareskins' and 'cart-horses' == 'carthorses'.
    """
    s = norm(x)
    s = re.sub(r"[\s\-]+", "", s)
    return s

def to_float(x):
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    if s == "":
        return None
    s = s.replace("\u00A0", " ").replace(" ", "")
    if "," in s and "." not in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def clear_range(ws, cell_range):
    for row in ws[cell_range]:
        for cell in row:
            cell.value = None


def find_sheet_by_keyword(wb, keyword):
    kw = keyword.lower()
    # prefer exact match if present
    for name in wb.sheetnames:
        if name.strip().lower() == kw:
            return name
    # else first containing match
    for name in wb.sheetnames:
        if kw in name.lower():
            return name
    return None


def key_in_sentence(key_norm, sentence):
    if not key_norm or sentence is None:
        return False
    sent = str(sentence).lower().replace("\u00A0", " ")
    sent = sent.replace("–", "-").replace("—", "-")
    parts = [re.escape(p) for p in re.split(r"[\s\-]+", key_norm) if p]
    if not parts:
        return False
    pattern = r"\b" + r"[\s\-]+".join(parts) + r"\b"
    return re.search(pattern, sent, flags=re.IGNORECASE) is not None


def is_total_sentence(x):
    return norm(x).startswith("total")


# ---------------- MAIN LOOP ----------------
for file_path in input_files:
    filename = os.path.basename(file_path)
    manor_name = filename.replace("_1409_6.xlsx", "")

    try:
        wb = load_workbook(file_path)

        manor_stock_name = f"{manor_name} Stock"
        if manor_stock_name not in wb.sheetnames:
            print(f"{filename}: no '{manor_stock_name}' sheet, skipping.")
            continue

        if "Stock" not in wb.sheetnames:
            print(f"{filename}: no 'Stock' sheet, skipping.")
            continue

        receipts_name = find_sheet_by_keyword(wb, "receipts")
        if receipts_name is None:
            print(f"{filename}: no Receipts sheet found (R will be 0).")

        ws_manor = wb[manor_stock_name]            # formatted stock
        ws_raw = wb["Stock"]                       # raw stock
        ws_receipts = wb[receipts_name] if receipts_name else None

        # ---- CLEAR TARGET AREAS (VALUES ONLY) ----
        clear_range(ws_manor, "B5:K31")
        clear_range(ws_manor, "R5:R31")
        clear_range(ws_manor, "O5:O31")
        clear_range(ws_manor, "B34:K47")  # wool & hides (B,C,H only for now)

        # ---- BUILD RAW LOOKUPS (KEYED BY RAW STOCK COL E, COMPACTED) ----
        raw_sum_F, raw_sum_G, raw_sum_H, raw_sum_I = {}, {}, {}, {}

        for rr in range(1, ws_raw.max_row + 1):
            k = key_id(ws_raw.cell(row=rr, column=5).value)  # raw E (category label)
            if not k:
                continue

            f = to_float(ws_raw.cell(row=rr, column=6).value)  # raw F
            g = to_float(ws_raw.cell(row=rr, column=7).value)  # raw G
            h = to_float(ws_raw.cell(row=rr, column=8).value)  # raw H
            i = to_float(ws_raw.cell(row=rr, column=9).value)  # raw I

            if f is not None:
                raw_sum_F[k] = raw_sum_F.get(k, 0.0) + f
            if g is not None:
                raw_sum_G[k] = raw_sum_G.get(k, 0.0) + g
            if h is not None:
                raw_sum_H[k] = raw_sum_H.get(k, 0.0) + h
            if i is not None:
                raw_sum_I[k] = raw_sum_I.get(k, 0.0) + i


        # ---- PRECOMPUTE SALE-OF-STOCK RECEIPT ROWS ----
        # Layout from your screenshot:
        # - Column A repeats "Sale of stock" on each item row
        # - Later a terminating row "Sale of stock total"
        sale_rows = []
        if ws_receipts is not None:
            for rr in range(1, ws_receipts.max_row + 1):
                a = norm(ws_receipts.cell(row=rr, column=1).value)
                if a.startswith("sale of stock total"):
                    break
                if a == "sale of stock":
                    sale_rows.append(rr)

        # ---- LIVESTOCK BLOCK A5:A30 -> FILL B,C,H,I + R (SALES) + O=R/H ----
        for r in range(5, 31):
            manor_label = ws_manor.cell(row=r, column=1).value  # A
            label_norm = norm(manor_label)
            k = key_id(manor_label)  # <-- FIX: must match the raw_sum_* dict keys

            if not k:
                continue
            if label_norm.endswith("total"):
                continue

            # B, C, H, I from raw stock (dicts are keyed by key_id)
            ws_manor.cell(row=r, column=2).value = raw_sum_F.get(k, 0)  # B
            ws_manor.cell(row=r, column=3).value = raw_sum_G.get(k, 0)  # C
            ws_manor.cell(row=r, column=8).value = raw_sum_I.get(k, 0)  # H (raw I)
            ws_manor.cell(row=r, column=9).value = raw_sum_H.get(k, 0)  # I (raw H)

            # R from receipts (Sale of stock): match label text in sentence (use label_norm, not key_id)
            sale_total = 0.0
            found_sale = False
            if ws_receipts is not None and sale_rows:
                for rr in sale_rows:
                    sentence = ws_receipts.cell(row=rr, column=8).value  # H sentence
                    if is_total_sentence(sentence):
                        continue
                    if not key_in_sentence(label_norm, sentence):
                        continue
                    pounds = to_float(ws_receipts.cell(row=rr, column=6).value)  # F pounds
                    if pounds is None:
                        continue
                    sale_total += pounds
                    found_sale = True

            ws_manor.cell(row=r, column=18).value = sale_total if found_sale else 0  # R

        # O = R / H (div0 -> 0), rows 5..30
        for r in range(5, 31):
            r_val = to_float(ws_manor.cell(row=r, column=18).value)  # R
            h_val = to_float(ws_manor.cell(row=r, column=8).value)   # H
            if r_val is None or h_val is None or h_val == 0:
                ws_manor.cell(row=r, column=15).value = 0  # O
            else:
                ws_manor.cell(row=r, column=15).value = r_val / h_val


        # ---- WOOL & HIDES BLOCK A34:A62 -> FILL B,C,H,I ----
        # Requirements:
        # - handle merged/blank labels in col A
        # - totals like "bareskins total" should match raw category "bareskins"
        # - ONLY fill totals where desired (e.g., row 59)

        ALLOWED_TOTAL_ROWS = {53, 59}  # add more rows here if needed

        last_label = None

        for r in range(34, 63):

            # Skip excluded subranges
            if 49 <= r <= 52 or 55 <= r <= 58:
                continue

            manor_label = ws_manor.cell(row=r, column=1).value

            # Handle merged/blank labels: carry forward last non-empty label
            if manor_label in (None, ""):
                manor_label = last_label
            else:
                last_label = manor_label

            if manor_label in (None, ""):
                continue  # still nothing usable

            label_norm = norm(manor_label)

            # If this is a "... total" row, strip "total" for matching
            if label_norm.endswith(" total"):
                base_norm = label_norm[:-6].strip()
                k = key_id(base_norm)
                # skip totals unless explicitly allowed
                if r not in ALLOWED_TOTAL_ROWS:
                    continue
            elif label_norm.endswith(" totals"):
                base_norm = label_norm[:-7].strip()
                k = key_id(base_norm)
                if r not in ALLOWED_TOTAL_ROWS:
                    continue
            else:
                k = key_id(label_norm)

            if not k:
                continue

            ws_manor.cell(row=r, column=2).value = raw_sum_F.get(k, 0)  # B
            ws_manor.cell(row=r, column=3).value = raw_sum_G.get(k, 0)  # C
            ws_manor.cell(row=r, column=8).value = raw_sum_I.get(k, 0)  # H (raw I)
            ws_manor.cell(row=r, column=9).value = raw_sum_H.get(k, 0)  # I (raw H = murrain)

            # ---- FINAL CLEANUP: clear structural header rows in hides block ----
            # Sloppy fix, shouldn't be filled in the first place. 
            clear_range(ws_manor, "B48:I48")  # "Hides" header
            clear_range(ws_manor, "B54:I54")  # "Bareskins" header


        # ---- POULTRY STOCK A67:A70 -> FILL R FROM RECEIPTS "ISSUES OF THE MANOR" + O=R/H ----

        # Optional clear (recommended)
        clear_range(ws_manor, "R67:R70")
        clear_range(ws_manor, "O67:O70")

        # Collect "Issues of the manor" receipt rows (A == Issues of the manor) until "... total"
        issues_rows = []
        if ws_receipts is not None:
            for rr in range(1, ws_receipts.max_row + 1):
                a = norm(ws_receipts.cell(row=rr, column=1).value)
                if a.startswith("issues of the manor total"):
                    break
                if a == "issues of the manor":
                    issues_rows.append(rr)

        # For each poultry key, sum matching Issues-of-the-manor lines by sentence in col H, value from col F
        for r in range(67, 71):
            manor_label = ws_manor.cell(row=r, column=1).value
            label_norm = norm(manor_label)
            if not label_norm or label_norm.endswith("total"):
                continue

            issue_total = 0.0
            found = False

            if ws_receipts is not None and issues_rows:
                for rr in issues_rows:
                    sentence = ws_receipts.cell(row=rr, column=8).value  # Receipts H
                    if is_total_sentence(sentence):
                        continue
                    if not key_in_sentence(label_norm, sentence):
                        continue

                    pounds = to_float(ws_receipts.cell(row=rr, column=6).value)  # Receipts F
                    if pounds is None:
                        continue

                    issue_total += pounds
                    found = True

            # Write poultry sales/issue pounds into Stock column R
            ws_manor.cell(row=r, column=18).value = issue_total if found else 0  # R

        # Compute O = R / H for poultry rows (div0 -> 0)
        for r in range(67, 71):
            r_val = to_float(ws_manor.cell(row=r, column=18).value)  # R
            h_val = to_float(ws_manor.cell(row=r, column=8).value)   # H
            if r_val is None or h_val is None or h_val == 0:
                ws_manor.cell(row=r, column=15).value = 0  # O
            else:
                ws_manor.cell(row=r, column=15).value = r_val / h_val

        # ---- MISC STOCK ROWS -> FILL B,C,H,I (SAME STRATEGY AS LIVESTOCK; RAW KEYED BY key_id) ----
        # Target rows: 75, 76, 82, 83, 84, 88, 89, 93, 99
        # Assumes raw_sum_F/raw_sum_G/raw_sum_H/raw_sum_I already built with keys = key_id(raw E)

        target_rows = [75, 76, 82, 83, 84, 88, 89, 93, 99]

        # Optional: clear just the target cells first (values only)
        for r in target_rows:
            clear_range(ws_manor, f"B{r}:K{r}")

        for r in target_rows:
            manor_label = ws_manor.cell(row=r, column=1).value  # A
            label_norm = norm(manor_label)
            k = key_id(manor_label)

            if not k:
                continue
            if label_norm.endswith("total"):
                continue

            # B <- raw F
            ws_manor.cell(row=r, column=2).value = raw_sum_F.get(k, 0)

            # C <- raw G
            ws_manor.cell(row=r, column=3).value = raw_sum_G.get(k, 0)

            # H <- raw I
            ws_manor.cell(row=r, column=8).value = raw_sum_I.get(k, 0)

            # I <- raw H
            ws_manor.cell(row=r, column=9).value = raw_sum_H.get(k, 0)


        wb.save(file_path)
        print(f"{filename}: updated {manor_stock_name} (livestock + wool/hides) via script 6.4.")

    except Exception as e:
        print(f"Error processing {filename}: {e}")


Found 56 workbooks
Adderbury_1409_6.xlsx: updated Adderbury Stock (livestock + wool/hides) via script 6.4.
Alresford_1409_6.xlsx: updated Alresford Stock (livestock + wool/hides) via script 6.4.
Alverstoke_1409_6.xlsx: updated Alverstoke Stock (livestock + wool/hides) via script 6.4.
Ashmansworth_1409_6.xlsx: updated Ashmansworth Stock (livestock + wool/hides) via script 6.4.
Beauworth_1409_6.xlsx: updated Beauworth Stock (livestock + wool/hides) via script 6.4.
Bentley_1409_6.xlsx: updated Bentley Stock (livestock + wool/hides) via script 6.4.
Bereleigh_1409_6.xlsx: updated Bereleigh Stock (livestock + wool/hides) via script 6.4.
BishopsFonthill_1409_6.xlsx: updated BishopsFonthill Stock (livestock + wool/hides) via script 6.4.
BishopsSutton_1409_6.xlsx: updated BishopsSutton Stock (livestock + wool/hides) via script 6.4.
BishopsWaltham_1409_6.xlsx: updated BishopsWaltham Stock (livestock + wool/hides) via script 6.4.
Bishopstone_1409_6.xlsx: updated Bishopstone Stock (livestock + woo

In [11]:
####################### SCRIPT 7 (ONE WORKING CELL: rename Overview/Stock, keep Stock_raw, use manor Stock for values, force formulas to 'Stock', reorder sheets) #######################

import os
import glob
import re
from pathlib import Path
from copy import copy as pycopy
from openpyxl import load_workbook
from openpyxl.cell.cell import MergedCell
from openpyxl.worksheet.views import Selection

# ---------- PATHS ----------
input_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script6\\"
output_base_path = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script7\\"
template_base_path = r"C:\Users\kubak\Documents\GitHub\student_assistant_manorial_records\archive"
template_filename = "Adderbury 1409_wl_format.xlsx"

Path(output_base_path).mkdir(parents=True, exist_ok=True)
template_path = os.path.join(template_base_path, template_filename)

TRAIL_PUNCT = (":", ";", ".", ",")

# ADD EDGE CASES HERE 
OVERVIEW_TO_RECEIPTS_MAP = {
    "rents of assize": "rents",
    "total of rents remaining clear total":"total of rents remaining",
    "pleas & perquisites": "perquisites of court",
    "pleas and perquisites": "perquisites of court",
    }
OVERVIEW_TO_EXPENSES_MAP = {
    # ---- Pleas / perquisites (most specific first) ----
    "pleas & perquisites total": "perquisites of court total",
    "pleas and perquisites total": "perquisites of court total",
    "pleas & perquisites": "perquisites of court",
    "pleas and perquisites": "perquisites of court",
    "perquisites total": "perquisites of court",
    "perquisites": "perquisites of court",

    # ---- Carts (specific BEFORE generic) ----
    "cost of carts and wagons": "cost of carts",
    "cost of carts": "carts",

    # ---- Other expenses ----
    "steward's expenses": "expenses of the steward",
    "expenses of the harvest": "cost of the harvest",
    "purchase of corn": "purchase of seed and liveries",
    "expenses of the harvest": "cost of the harvest"
}

GRAIN_MAP = {
    "wheat": "wheat", "rye": "rye", "dredge": "dredge", "curall": "curall",
    "barley": "barley", "oats": "oats", "beans": "beans", "peas": "peas", "vetches": "vetches",
}

# ---------- HELPERS ----------
def normalize_label(s):
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("\u00A0", " ")
    s = " ".join(s.split()).strip().lower()
    while s.endswith(TRAIL_PUNCT):
        s = s[:-1].strip()
    return s

def key_id(x):
    """
    Robust join key:
    - lowercase, collapse whitespace
    - treat hyphens/spaces as equivalent
    - strip trailing punctuation
    Example: 'Cart-horses' == 'cart horses' == 'cart-horses'
    """
    s = normalize_label(x)
    s = re.sub(r"[\s\-]+", "", s)
    return s

def strip_total(s):
    s = normalize_label(s)
    if s.endswith(" totals"):
        return s[:-7].strip()
    if s.endswith(" total"):
        return s[:-6].strip()
    return s

def overview_to_key(overview_label, mapping_dict):
    s = strip_total(overview_label)
    for phrase, repl in mapping_dict.items():
        if s.startswith(phrase):
            return repl
    return s

def build_last_row_index_multi(ws, start_row=3, label_col=1):
    last = {}
    for r in range(start_row, ws.max_row + 1):
        v = ws.cell(row=r, column=label_col).value
        key1 = normalize_label(v)
        if not key1:
            continue
        key2 = strip_total(key1)
        last[key1] = r
        last[key2] = r
    return last

def find_last_row_startswith(ws, key, start_row=3, label_col=1):
    key = normalize_label(key)
    if not key:
        return None
    last = None
    for r in range(start_row, ws.max_row + 1):
        v = normalize_label(ws.cell(row=r, column=label_col).value)
        if v.startswith(key):
            last = r
    return last

def find_sheet_by_keyword(wb, keyword):
    if keyword in wb.sheetnames:
        return keyword
    kw = keyword.lower()
    for name in wb.sheetnames:
        if kw in name.lower():
            return name
    return None

def find_grange_sheet(wb):
    for nm in wb.sheetnames:
        ln = nm.lower()
        if "issues" in ln and "grange" in ln:
            return nm
    return None

def to_number(x):
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        s = x.strip().replace("\u00A0", " ").replace(" ", "")
        if "," in s and "." not in s:
            s = s.replace(",", ".")
        try:
            return float(s)
        except ValueError:
            return None
    return None

def num_or_zero(x):
    n = to_number(x)
    return 0.0 if n is None else n

def clean_excel_formula(v):
    if isinstance(v, str) and v.startswith("=+"):
        return "=" + v[2:]
    return v

def rename_sheet_if_exists(wb, old_name, new_name):
    if old_name in wb.sheetnames and new_name not in wb.sheetnames:
        wb[old_name].title = new_name

def remove_sheet_if_exists(wb, name):
    if name in wb.sheetnames:
        wb.remove(wb[name])

def find_manor_stock_sheet(wb, manor_name):
    exact = f"{manor_name} Stock"
    if exact in wb.sheetnames:
        return exact
    mn = manor_name.lower()
    for nm in wb.sheetnames:
        ln = nm.lower()
        if "stock" in ln and mn in ln:
            return nm
    return None

def reorder_sheets(wb, preferred_order):
    # preferred_order: list of names (some may not exist)
    name_to_ws = {ws.title: ws for ws in wb.worksheets}
    first = [name_to_ws[n] for n in preferred_order if n in name_to_ws]
    rest = [ws for ws in wb.worksheets if ws.title not in preferred_order]
    wb._sheets = first + rest

def find_first_row_exact_norm(ws, target_norm, col=1, start_row=1):
    """
    Return the first row index >= start_row where the normalized cell value in `col`
    equals `target_norm` (which is assumed already normalized).
    """
    if ws is None:
        return None
    for r in range(start_row, ws.max_row + 1):
        if normalize_label(ws.cell(row=r, column=col).value) == target_norm:
            return r
    return None

def find_last_row_exact_norm(ws, target_norm, col=1, start_row=1):
    """Return the last row >= start_row where normalized ws[col] equals target_norm."""
    last = None
    for r in range(start_row, ws.max_row + 1):
        if normalize_label(ws.cell(row=r, column=col).value) == target_norm:
            last = r
    return last

def set_overview_cell_as_expenses_ref(ws_overview, wb_out, expenses_sheet_name, ov_cell_addr, target_label_raw):
    """
    Set Overview cell to reference Expenses!F<matched_row>.
    If the label cannot be found, write 0 instead of leaving the cell blank.
    """
    if not expenses_sheet_name or expenses_sheet_name not in wb_out.sheetnames:
        ws_overview[ov_cell_addr].value = 0
        return

    ws_exp = wb_out[expenses_sheet_name]
    target_norm = normalize_label(target_label_raw)

    match_row = None
    for r in range(1, ws_exp.max_row + 1):
        if normalize_label(ws_exp.cell(row=r, column=1).value) == target_norm:
            match_row = r  # last match wins

    if not match_row:
        ws_overview[ov_cell_addr].value = 0
        return

    ws_overview[ov_cell_addr].value = f"='{expenses_sheet_name}'!F{match_row}"

# ---------- COPY ROUTINE (VALUES + FORMATTING) ----------
def copy_sheet_full(src_ws, dest_ws):
    dest_ws.freeze_panes = src_ws.freeze_panes
    if src_ws.auto_filter and src_ws.auto_filter.ref:
        dest_ws.auto_filter.ref = src_ws.auto_filter.ref

    for col_letter, dim in src_ws.column_dimensions.items():
        d = dest_ws.column_dimensions[col_letter]
        d.width = dim.width
        d.hidden = dim.hidden
        d.outlineLevel = dim.outlineLevel
        d.collapsed = dim.collapsed

    for row_idx, dim in src_ws.row_dimensions.items():
        d = dest_ws.row_dimensions[row_idx]
        d.height = dim.height
        d.hidden = dim.hidden
        d.outlineLevel = dim.outlineLevel
        d.collapsed = dim.collapsed

    for merged_range in src_ws.merged_cells.ranges:
        dest_ws.merge_cells(str(merged_range))

    for row in src_ws.iter_rows():
        for c in row:
            if isinstance(c, MergedCell):
                continue
            dc = dest_ws.cell(row=c.row, column=c.column, value=c.value)

            if c.has_style:
                dc._style = pycopy(c._style)
                dc.font = pycopy(c.font)
                dc.border = pycopy(c.border)
                dc.fill = pycopy(c.fill)
                dc.number_format = c.number_format
                dc.protection = pycopy(c.protection)
                dc.alignment = pycopy(c.alignment)

            if c.hyperlink:
                dc._hyperlink = pycopy(c.hyperlink)
            if c.comment:
                dc.comment = pycopy(c.comment)

    dest_ws.page_margins = pycopy(src_ws.page_margins)
    dest_ws.page_setup = pycopy(src_ws.page_setup)
    dest_ws.print_options = pycopy(src_ws.print_options)

# ---------- PROCESS ALL SCRIPT 6 FILES ----------
pattern = os.path.join(input_base_path, "*.xlsx")
input_files = sorted(glob.glob(pattern))
print(f"Found {len(input_files)} input workbooks")

for input_file in input_files:
    src_filename = os.path.basename(input_file)
    base_no_ext = os.path.splitext(src_filename)[0]
    manor_name = base_no_ext.split("_")[0]

    output_filename = f"{manor_name}_7.xlsx"
    output_path = os.path.join(output_base_path, output_filename)

    # 1) template -> output base
    wb_out = load_workbook(template_path, data_only=False)
    if "Adderbury Overview" not in wb_out.sheetnames:
        wb_out.close()
        raise RuntimeError(f"Template error: 'Adderbury Overview' sheet not found in {template_filename}")

    ws_overview = wb_out["Adderbury Overview"]
    ws_overview.title = f"{manor_name} Overview - Completed"
    ws_overview["A1"].value = f"{manor_name.upper()} OVERVIEW - COMPLETED"

    for sh in list(wb_out.sheetnames):
        if sh.startswith("Adderbury ") and sh != ws_overview.title:
            wb_out.remove(wb_out[sh])

    # 2) copy input workbook sheets (format preserved) into output
    wb_in = load_workbook(input_file, data_only=False)
    wb_in_vals = load_workbook(input_file, data_only=True)

    for src_ws in wb_in.worksheets:
        src_title = src_ws.title
        if src_title in wb_out.sheetnames:
            wb_out.remove(wb_out[src_title])
        dest_ws = wb_out.create_sheet(src_title)
        copy_sheet_full(src_ws, dest_ws)

    # ---------- POST-COPY CLEANUP + RENAMES + ORDER ----------
    # Remove generic Overview sheet from input (keep the template-derived one)
    remove_sheet_if_exists(wb_out, "Overview")

    # Make raw stock non-canonical: Stock -> Stock_raw
    rename_sheet_if_exists(wb_out, "Stock", "Stock_raw")

    # Rename formatted manor stock -> Stock
    manor_stock_out = find_manor_stock_sheet(wb_out, manor_name)
    if manor_stock_out:
        if "Stock" in wb_out.sheetnames:
            wb_out.remove(wb_out["Stock"])
        wb_out[manor_stock_out].title = "Stock"

    # Rename template overview -> Overview
    if f"{manor_name} Overview - Completed" in wb_out.sheetnames:
        remove_sheet_if_exists(wb_out, "Overview")
        wb_out[f"{manor_name} Overview - Completed"].title = "Overview"

    # Safety: remove any leftover Adderbury Overview
    remove_sheet_if_exists(wb_out, "Adderbury Overview")

    # Refresh handle after rename
    ws_overview = wb_out["Overview"]

    # Reorder sheets: Overview, Receipts, Expenses, Stock, rest
    receipts_out_name = find_sheet_by_keyword(wb_out, "Receipts") or find_sheet_by_keyword(wb_out, "receipt")
    expenses_out_name = find_sheet_by_keyword(wb_out, "Expenses") or find_sheet_by_keyword(wb_out, "expense")
    preferred = [n for n in ["Overview", receipts_out_name, expenses_out_name, "Stock"] if n]
    reorder_sheets(wb_out, preferred)

    # ---------- RECEIPTS (B4:B22) ----------
    receipts_out_name = find_sheet_by_keyword(wb_out, "Receipts") or find_sheet_by_keyword(wb_out, "receipt")
    receipts_in_name  = find_sheet_by_keyword(wb_in_vals, "Receipts") or find_sheet_by_keyword(wb_in_vals, "receipt")
    if receipts_out_name and receipts_in_name:
        ws_receipts_out = wb_out[receipts_out_name]
        ws_receipts_val = wb_in_vals[receipts_in_name]
        last_rows = build_last_row_index_multi(ws_receipts_out, start_row=3, label_col=1)

        for r in range(4, 23):
            key = overview_to_key(ws_overview.cell(row=r, column=1).value, OVERVIEW_TO_RECEIPTS_MAP)
            if not key:
                continue

            match_row = last_rows.get(key)
            if match_row is None and key.endswith("s"):
                match_row = last_rows.get(key[:-1])
            if match_row is None:
                match_row = find_last_row_startswith(ws_receipts_out, key, start_row=3, label_col=1)

            if match_row is None:
                ws_overview.cell(row=r, column=2).value = None
                continue

            cached_val = ws_receipts_val.cell(row=match_row, column=6).value
            ws_overview.cell(row=r, column=2).value = cached_val if cached_val is not None else clean_excel_formula(
                ws_receipts_out.cell(row=match_row, column=6).value
            )

    # ---------- EXPENSES (B30:B43) ----------
    expenses_out_name = find_sheet_by_keyword(wb_out, "Expenses") or find_sheet_by_keyword(wb_out, "expense")
    expenses_in_name  = find_sheet_by_keyword(wb_in_vals, "Expenses") or find_sheet_by_keyword(wb_in_vals, "expense")
    if expenses_out_name and expenses_in_name:
        ws_expenses_out = wb_out[expenses_out_name]
        ws_expenses_val = wb_in_vals[expenses_in_name]
        last_rows_exp = build_last_row_index_multi(ws_expenses_out, start_row=3, label_col=1)

        for r in range(30, 44):
            key = overview_to_key(ws_overview.cell(row=r, column=1).value, OVERVIEW_TO_EXPENSES_MAP)
            if not key:
                continue

            match_row = last_rows_exp.get(key)
            if match_row is None and key.endswith("s"):
                match_row = last_rows_exp.get(key[:-1])
            if match_row is None:
                match_row = find_last_row_startswith(ws_expenses_out, key, start_row=3, label_col=1)

            if match_row is None:
                ws_overview.cell(row=r, column=2).value = None
                continue

            cached_val = ws_expenses_val.cell(row=match_row, column=6).value
            ws_overview.cell(row=r, column=2).value = cached_val if cached_val is not None else clean_excel_formula(
                ws_expenses_out.cell(row=match_row, column=6).value
            )

    # ---------- ISSUES OF THE GRANGE → OVERVIEW (A60:E73) ----------

    grange_name = "Issues of the Grange"
    if grange_name in wb_out.sheetnames:
        ws_grange = wb_out[grange_name]

        # Build lookup dicts keyed on column A
        grange_C = {}   # account quantity
        grange_D = {}   # bought quantity
        grange_P = {}   # gross value (£)
        grange_H = {}   # bought quantity (alt)
        grange_N = {}   # price per quarter

        for r in range(2, ws_grange.max_row + 1):
            key = normalize_label(ws_grange.cell(row=r, column=1).value)
            if not key:
                continue

            grange_C[key] = to_number(ws_grange.cell(row=r, column=3).value)  # C
            grange_D[key] = to_number(ws_grange.cell(row=r, column=4).value)  # D
            grange_P[key] = to_number(ws_grange.cell(row=r, column=16).value) # P
            grange_H[key] = to_number(ws_grange.cell(row=r, column=8).value)  # H
            grange_N[key] = to_number(ws_grange.cell(row=r, column=14).value) # N

        # Populate Overview rows 62..70
        for ov_row in range(62, 71):
            key = normalize_label(ws_overview.cell(row=ov_row, column=1).value)
            if not key:
                continue

            # Column B: account quantity
            ws_overview.cell(row=ov_row, column=2).value = grange_C.get(key, 0)

            # Column C: bought quantity
            ws_overview.cell(row=ov_row, column=3).value = grange_D.get(key, 0)

            # Column D: gross production value (£)
            gross = grange_P.get(key, 0)
            ws_overview.cell(row=ov_row, column=4).value = gross

            # Column E: net production value (£) = gross − (bought × price)
            bought_qty = grange_H.get(key, 0)
            price = grange_N.get(key, 0)

            if gross is None:
                ws_overview.cell(row=ov_row, column=5).value = 0
            else:
                ws_overview.cell(row=ov_row, column=5).value = gross - (bought_qty * price)

    else:
        print("No 'Issues of the Grange' sheet found; Overview A60:E73 not populated.")


    # ---------------- OVERVIEW B73: SALES VALUE (sum over keys of (K * N)) ----------------
    # Keys are taken from Overview A62:A73 (inclusive).
    # For matching keys in "Issues of the Grange", sum (col K * col N) and write to Overview B73.

    grange_name = "Issues of the Grange"
    if grange_name in wb_out.sheetnames:
        ws_grange = wb_out[grange_name]

        # Build a key set from Overview A62:A73
        keyset = set()
        for ov_row in range(62, 74):  # 62..73 inclusive
            k = normalize_label(ws_overview.cell(row=ov_row, column=1).value)
            if k:
                keyset.add(k)

        total_sales_value = 0.0

        # Scan Issues of the Grange once; sum K*N for matching keys
        for r in range(2, ws_grange.max_row + 1):
            k = normalize_label(ws_grange.cell(row=r, column=1).value)  # A = key
            if not k or k not in keyset:
                continue

            k_val = to_number(ws_grange.cell(row=r, column=11).value)  # K
            n_val = to_number(ws_grange.cell(row=r, column=14).value)  # N
            if k_val is None or n_val is None:
                continue

            total_sales_value += (k_val * n_val)

        # Write into Overview B73
        ws_overview.cell(row=73, column=5).value = total_sales_value
    else:
        print("No 'Issues of the Grange' sheet found; Overview B73 not populated.")

    # ---------- LIVESTOCK ----------
    # Requirement:
    # - Match by key in col A
    # - Overview B <- Stock B
    # - Overview C <- Stock L
    # IMPORTANT:
    # - Read VALUES from the INPUT manor stock sheet using wb_in_vals (data_only=True),
    #   because the OUTPUT formatted Stock sheet may contain formulas and openpyxl won't calculate them.
    # - Use OUTPUT 'Stock' only as a fallback formula reference when a cached value is missing.

    manor_stock_in_name = find_manor_stock_sheet(wb_in_vals, manor_name)

    if not manor_stock_in_name:
        print(f"No manor stock sheet found in input for {manor_name}; skipping livestock fill.")
    else:
        ws_stock_val = wb_in_vals[manor_stock_in_name]  # values (data_only=True)

        # Optional: output Stock for fallback formulas
        ws_stock_out = wb_out["Stock"] if "Stock" in wb_out.sheetnames else None

        # Build lookup from Stock rows 5..30 with merged-cell-safe label propagation
        stock_row_for_key = {}
        stock_B = {}  # begin stock (B)
        stock_L = {}  # end stock (L)

        last_key = None
        for rr in range(5, 31):
            raw_label = ws_stock_val.cell(row=rr, column=1).value  # A
            k = key_id(raw_label) if raw_label not in (None, "") else None

            # Handle merged/blank labels by carrying forward the last non-empty key
            if k:
                last_key = k
            else:
                k = last_key

            if not k:
                continue

            stock_row_for_key[k] = rr
            stock_B[k] = ws_stock_val.cell(row=rr, column=2).value    # B
            stock_L[k] = ws_stock_val.cell(row=rr, column=12).value   # L

        # Fill Overview rows 87..112
        for ov_row in range(87, 113):
            k = key_id(ws_overview.cell(row=ov_row, column=1).value)  # Overview A
            if not k:
                continue

            rr = stock_row_for_key.get(k)

            b_val = stock_B.get(k, None)
            c_val = stock_L.get(k, None)

            # If the join fails, do NOT silently write 0; leave blank so it's obvious.
            # (You can change to 0 if you prefer.)
            if rr is None:
                ws_overview.cell(row=ov_row, column=2).value = None
                ws_overview.cell(row=ov_row, column=3).value = None
                continue

            # Force formulas to use canonical Stock sheet name (never Adderbury Stock)
            # Choose the correct columns for your template:
            # - end quantity is in Stock!L
            # - unit price is in Stock!O (adjust if your unit price column is different)

            if rr is not None and "Stock" in wb_out.sheetnames:
                # Overview F: end stock value
                ws_overview.cell(row=ov_row, column=6).value = f"='Stock'!L{rr}*'Stock'!O{rr}"
                # Overview G: change in stock value
                ws_overview.cell(row=ov_row, column=7).value = f"=D{ov_row}*'Stock'!O{rr}"


            # Write cached numeric values when present
            if b_val is not None and b_val != "":
                ws_overview.cell(row=ov_row, column=2).value = b_val
            else:
                # Fallback: point to OUTPUT Stock B if available
                ws_overview.cell(row=ov_row, column=2).value = (
                    f"='Stock'!B{rr}" if ws_stock_out is not None else None
                )

            if c_val is not None and c_val != "":
                ws_overview.cell(row=ov_row, column=3).value = c_val
            else:
                # Fallback: point to OUTPUT Stock L if available
                ws_overview.cell(row=ov_row, column=3).value = (
                    f"='Stock'!L{rr}" if ws_stock_out is not None else None
                )

            # Overview D = C - B (leave as formula if either side non-numeric)
            nb, nc = to_number(ws_overview.cell(row=ov_row, column=2).value), to_number(ws_overview.cell(row=ov_row, column=3).value)
            if nb is not None and nc is not None:
                ws_overview.cell(row=ov_row, column=4).value = nc - nb
            else:
                ws_overview.cell(row=ov_row, column=4).value = f"=C{ov_row}-B{ov_row}"


    # ---------------- POULTRY (Overview rows 151..154) ----------------
    # Keys: Overview!A151:A154
    # Source VALUES: input manor stock sheet (wb_in_vals), rows 67..70
    # Fill:
    #   Overview B = Stock!B
    #   Overview C = Stock!B - Stock!M
    #   Overview D = B - C
    #   Overview F = Stock!Q
    #
    # Fallback (if cached values missing): write formulas pointing to OUTPUT 'Stock' sheet (canonical name)

    manor_stock_in_name = find_manor_stock_sheet(wb_in_vals, manor_name)
    if not manor_stock_in_name:
        print(f"No manor stock sheet found in input for {manor_name}; skipping poultry fill (Overview 151..154).")
    else:
        ws_stock_val = wb_in_vals[manor_stock_in_name]  # values (data_only=True)
        has_out_stock = ("Stock" in wb_out.sheetnames)

        # Build lookup from input Stock rows 67..70, merged-label-safe
        stock_row_for_key = {}
        stock_B = {}
        stock_M = {}
        stock_Q = {}

        last_key = None
        for rr in range(67, 71):  # 67..70 inclusive
            raw_label = ws_stock_val.cell(row=rr, column=1).value  # A
            k = key_id(raw_label) if raw_label not in (None, "") else None

            if k:
                last_key = k
            else:
                k = last_key

            if not k:
                continue

            stock_row_for_key[k] = rr
            stock_B[k] = ws_stock_val.cell(row=rr, column=2).value    # B
            stock_M[k] = ws_stock_val.cell(row=rr, column=13).value   # M
            stock_Q[k] = ws_stock_val.cell(row=rr, column=17).value   # Q

        # Populate Overview rows 151..154
        for ov_row in range(151, 155):
            k = key_id(ws_overview.cell(row=ov_row, column=1).value)  # Overview A
            if not k:
                continue

            rr = stock_row_for_key.get(k)
            if rr is None:
                # Do not silently write 0s; leave blank so join issues are visible
                ws_overview.cell(row=ov_row, column=2).value = None
                ws_overview.cell(row=ov_row, column=3).value = None
                ws_overview.cell(row=ov_row, column=4).value = None
                ws_overview.cell(row=ov_row, column=6).value = None
                continue

            b_raw = stock_B.get(k, None)
            m_raw = stock_M.get(k, None)
            q_raw = stock_Q.get(k, None)

            nb = to_number(b_raw)
            nm = to_number(m_raw)

            # Overview B
            if b_raw not in (None, ""):
                ws_overview.cell(row=ov_row, column=2).value = b_raw
            else:
                ws_overview.cell(row=ov_row, column=2).value = f"='Stock'!B{rr}" if has_out_stock else None

            # Overview C = B - M
            if nb is not None and nm is not None:
                c_val = nb - nm
                ws_overview.cell(row=ov_row, column=3).value = c_val
            else:
                # fallback formula (canonical Stock)
                ws_overview.cell(row=ov_row, column=3).value = f"='Stock'!B{rr}-'Stock'!M{rr}" if has_out_stock else None

            # Overview D = B - C  (this simplifies to M, but keep your stated logic)
            ws_overview.cell(row=ov_row, column=4).value = f"=B{ov_row}-C{ov_row}"

            # Overview F = Stock Q
            q_num = to_number(q_raw)
            if q_raw not in (None, ""):
                ws_overview.cell(row=ov_row, column=6).value = q_num if q_num is not None else q_raw
            else:
                ws_overview.cell(row=ov_row, column=6).value = f"='Stock'!Q{rr}" if has_out_stock else None

    # ---------------- POULTRY PRODUCTION (Overview rows 159..161) ----------------
    # Keys: Overview!A159:A161
    # Source: Stock sheet rows 75..77
    # Fill:
    #   Overview B = Stock!B
    #   Overview C = Stock!B - Stock!M
    #   Overview D = Overview B - Overview C
    #   Overview F = Stock!Q

    if "Stock" not in wb_out.sheetnames:
        print("No 'Stock' sheet found; skipping poultry production fill (Overview 159..161).")
    else:
        ws_stock = wb_out["Stock"]

        # Build lookup from Stock rows 75..77
        stock_B = {}
        stock_M = {}
        stock_Q = {}

        for rr in range(75, 78):  # 75..77 inclusive
            k = key_id(ws_stock.cell(row=rr, column=1).value)  # Stock A
            if not k:
                continue
            stock_B[k] = ws_stock.cell(row=rr, column=2).value   # B
            stock_M[k] = ws_stock.cell(row=rr, column=13).value  # M
            stock_Q[k] = ws_stock.cell(row=rr, column=17).value  # Q

        # Populate Overview rows 159..161
        for ov_row in range(159, 162):
            k = key_id(ws_overview.cell(row=ov_row, column=1).value)  # Overview A
            if not k:
                continue

            b_val = num_or_zero(stock_B.get(k, None))
            m_val = num_or_zero(stock_M.get(k, None))
            q_val = stock_Q.get(k, None)

            # Overview B
            ws_overview.cell(row=ov_row, column=2).value = b_val

            # Overview C = Stock B - Stock M
            c_val = b_val - m_val
            ws_overview.cell(row=ov_row, column=3).value = c_val

            # Overview D = B - C
            ws_overview.cell(row=ov_row, column=4).value = b_val - c_val

            # Overview F = Stock Q
            q_num = to_number(q_val)
            ws_overview.cell(row=ov_row, column=6).value = q_num if q_num is not None else q_val

    # ---------------- DAIRY PRODUCTION (Overview rows 166..168) ----------------
    # Keys: Overview!A166:A168
    # Source: Stock sheet rows 88, 89, and 93 ONLY (explicitly excluding 90..92)
    # Fill logic:
    #   Overview B = Stock!B
    #   Overview C = Stock!B − Stock!M
    #   Overview D = Overview B − Overview C
    #   Overview F = Stock!Q

    if "Stock" not in wb_out.sheetnames:
        print("No 'Stock' sheet found; skipping dairy production fill (Overview 166..168).")
    else:
        ws_stock = wb_out["Stock"]

        # Explicit stock rows to use for dairy
        dairy_stock_rows = [88, 89, 93]

        stock_B = {}
        stock_M = {}
        stock_Q = {}

        # Build lookup from selected Stock rows
        for rr in dairy_stock_rows:
            k = key_id(ws_stock.cell(row=rr, column=1).value)  # Stock column A
            if not k:
                continue
            stock_B[k] = ws_stock.cell(row=rr, column=2).value   # B
            stock_M[k] = ws_stock.cell(row=rr, column=13).value  # M
            stock_Q[k] = ws_stock.cell(row=rr, column=17).value  # Q

        # Populate Overview rows 166..168
        for ov_row in range(166, 169):
            k = key_id(ws_overview.cell(row=ov_row, column=1).value)  # Overview column A
            if not k:
                continue

            b_val = num_or_zero(stock_B.get(k, None))
            m_val = num_or_zero(stock_M.get(k, None))
            q_val = stock_Q.get(k, None)

            # Overview B
            ws_overview.cell(row=ov_row, column=2).value = b_val

            # Overview C = Stock B − Stock M
            c_val = b_val - m_val
            ws_overview.cell(row=ov_row, column=3).value = c_val

            # Overview D = B − C
            ws_overview.cell(row=ov_row, column=4).value = b_val - c_val

            # Overview F = Stock Q
            q_num = to_number(q_val)
            ws_overview.cell(row=ov_row, column=6).value = q_num if q_num is not None else q_val

    # ---------------- WOOL + HIDES (Overview) from Stock ----------------
    # Assumes:
    # - ws_overview = wb_out["Overview"]
    # - "Stock" sheet exists in wb_out
    # - key_id(), num_or_zero() exist

    if "Stock" not in wb_out.sheetnames:
        print("No 'Stock' sheet found; skipping wool/hides fills.")
    else:
        ws_stock = wb_out["Stock"]

        def fill_overview_BCD_from_stock(ov_start, ov_end, stock_start, stock_end):
            # Build lookup from Stock A within the specified row window
            stock_B = {}
            stock_C = {}
            for rr in range(stock_start, stock_end + 1):
                k = key_id(ws_stock.cell(row=rr, column=1).value)  # Stock A
                if not k:
                    continue
                stock_B[k] = ws_stock.cell(row=rr, column=2).value  # Stock B
                stock_C[k] = ws_stock.cell(row=rr, column=3).value  # Stock C

            # Populate Overview rows
            for ov_row in range(ov_start, ov_end + 1):
                k = key_id(ws_overview.cell(row=ov_row, column=1).value)  # Overview A
                if not k:
                    continue

                b_val = num_or_zero(stock_B.get(k, None))
                c_val = num_or_zero(stock_C.get(k, None))

                ws_overview.cell(row=ov_row, column=2).value = b_val          # Overview B
                ws_overview.cell(row=ov_row, column=3).value = c_val          # Overview C
                ws_overview.cell(row=ov_row, column=4).value = b_val - c_val  # Overview D = B - C

        # Wool: Overview 118:122 <- Stock 35:38
        fill_overview_BCD_from_stock(118, 122, 35, 38)

        # Hides part 1: Overview 127:133 <- Stock 43:47
        fill_overview_BCD_from_stock(127, 133, 43, 47)

        # Hides part 2: Overview 143:146 <- Stock 59:62
        fill_overview_BCD_from_stock(143, 146, 59, 62)

        # NEW: single-row fill for Overview row 138 only -> This is unbelievably sloppy. 
        fill_overview_BCD_from_stock(138, 138, 43, 47)

    # ---------------- OVERVIEW SUMMARY ROWS: ISSUES OF THE GRANGE ----------------
    # B71 = sum of gross production values (D62:D70)
    # B72 = sum of net production values   (E62:E70)

    ws_overview.cell(row=71, column=5).value = "=SUM(D62:D70)"
    ws_overview.cell(row=72, column=5).value = "=SUM(E62:E70)"

    # ---------------- SUMMARY ROWS (Overview) ----------------
    ws_overview["B23"].value = "=SUM(B5:B11)"
    ws_overview["B24"].value = "=SUM(B4:B11)"

    # ---------- OVERVIEW B25/B26: direct pull from Receipts by exact label match (col A -> col F) ----------
    # --- Row 25: reference Receipts!F<row> by label match in col A
    target_25 = normalize_label("Total of all receipts recorded in accounts (calculated here)")
    r25 = find_first_row_exact_norm(ws_receipts_out, target_25, col=1, start_row=1)
    ws_overview["B25"].value = f"='{receipts_out_name}'!F{r25}" if r25 else None

    # --- Row 26: reference Receipts!F<row> by label match in col A
    target_26 = normalize_label("Total of all receipts (calculated here) incl arrears")
    r26 = find_first_row_exact_norm(ws_receipts_out, target_26, col=1, start_row=1)
    ws_overview["B26"].value = f"='{receipts_out_name}'!F{r26}" if r26 else None

    # ---------------- ADDITIONAL SUMMARY CELLS (Overview) ----------------
    ws_overview["G113"].value = "=SUM(G87:G112)"

    # F114: sum Stock!T5:T30 (force 'Stock')
    ws_overview["F114"].value = "=SUM('Stock'!T5:T30)" if "Stock" in wb_out.sheetnames else None
    ws_overview["F115"].value = "=SUM('Stock'!R5:R30)" if "Stock" in wb_out.sheetnames else None
    ws_overview["F156"].value = "=SUM('Stock'!R67:R70)" if "Stock" in wb_out.sheetnames else None
    ws_overview["F162"].value = "=SUM(F159:F160)"
    ws_overview["F169"].value = "=SUM(F166:F168)"
    ws_overview["F123"].value = "=SUM(F118:F122)"
    ws_overview["B175"].value = "=F169+F162+F155+F147+F123+F115+E72"

    # ---------------- SUMMARY ROWS: EXPENSES (Overview) ----------------
    # Fill Overview B44:B47 from Expenses totals (exact label match in col A, take value from col F)

    expenses_out_name = find_sheet_by_keyword(wb_out, "Expenses") or find_sheet_by_keyword(wb_out, "expense")
    expenses_in_name  = find_sheet_by_keyword(wb_in_vals, "Expenses") or find_sheet_by_keyword(wb_in_vals, "expense")

    def find_last_row_exact(ws, target_norm, col=1):
        """Return the last row index where ws[col] equals target_norm (normalized)."""
        last = None
        for r in range(1, ws.max_row + 1):
            if normalize_label(ws.cell(row=r, column=col).value) == target_norm:
                last = r
        return last

    def write_overview_from_expenses_label(ov_row, target_label_norm):
        """
        Write Overview B{ov_row} from Expenses row whose col A matches target_label_norm (normalized),
        using cached numeric value from wb_in_vals when possible.
        """
        if not (expenses_out_name and expenses_in_name):
            ws_overview.cell(row=ov_row, column=2).value = None
            return

        ws_exp_out = wb_out[expenses_out_name]
        ws_exp_val = wb_in_vals[expenses_in_name]

        match_row = find_last_row_exact(ws_exp_out, target_label_norm, col=1)
        if not match_row:
            ws_overview.cell(row=ov_row, column=2).value = None
            return

        cached_val = ws_exp_val.cell(row=match_row, column=6).value  # F
        ws_overview.cell(row=ov_row, column=2).value = (
            cached_val if cached_val is not None
            else clean_excel_formula(ws_exp_out.cell(row=match_row, column=6).value)
        )

    # Core totals block (B44:B47)
    set_overview_cell_as_expenses_ref(
        ws_overview, wb_out, expenses_out_name, "B44",
        "Total of all expenses per account"
    )

    set_overview_cell_as_expenses_ref(
        ws_overview, wb_out, expenses_out_name, "B45",
        "Total of all expenses (calculated here)"
    )
    write_overview_from_expenses_label(46, "total fixed investment")
    write_overview_from_expenses_label(47, "total demesne fixed investment")
    write_overview_from_expenses_label(50, "and he owes")
    write_overview_from_expenses_label(52, "allowances")
    # Overview row 53 should pull from "cash deliveries total" (NOT the narrative line)
    write_overview_from_expenses_label(53, "cash deliveries total")

    # ---------- UNFREEZE PANES + NORMALIZE SHEET VIEWS ----------
    for ws in wb_out.worksheets:
        ws.freeze_panes = None
        sv = ws.sheet_view
        sv.workbookViewId = 0
        sv.topLeftCell = "A1"
        sv.zoomScale = None
        sv.zoomScaleNormal = None
        sv.zoomScalePageLayoutView = None
        sv.zoomScaleSheetLayoutView = None
        sv.view = None
        sv.rightToLeft = None
        sv.tabSelected = None
        sv.selection = [Selection(activeCell="A1", sqref="A1")]

    # ---------- SAVE ----------
    wb_out.calculation.fullCalcOnLoad = True
    wb_out.active = 0
    wb_out.save(output_path)
    wb_out.close()

    print(f"Created: {output_filename}")


Found 56 input workbooks
Created: Adderbury_7.xlsx
Created: Alresford_7.xlsx
Created: Alverstoke_7.xlsx
Created: Ashmansworth_7.xlsx
Created: Beauworth_7.xlsx
Created: Bentley_7.xlsx
Created: Bereleigh_7.xlsx
Created: BishopsFonthill_7.xlsx
Created: BishopsSutton_7.xlsx
Created: BishopsWaltham_7.xlsx
Created: Bishopstone_7.xlsx
Created: Bitterne_7.xlsx
Created: Brightwell_7.xlsx
Created: Burghclere_7.xlsx
Created: Cheriton_7.xlsx
Created: Crawley_7.xlsx
Created: Culham_7.xlsx
Created: DowntonBorough_7.xlsx
Created: Downton_7.xlsx
Created: Droxford_7.xlsx
Created: EastKnoyle_7.xlsx
Created: EastMeon_7.xlsx
Created: Ecchinswell_7.xlsx
Created: Esher_7.xlsx
Created: Farnham_7.xlsx
Created: Gosport_7.xlsx
Created: Hambledon_7.xlsx
Created: Harwell_7.xlsx
Created: Highclere_7.xlsx
Created: HindonBorough_7.xlsx
Created: Holway_7.xlsx
Created: Ivinghoe_7.xlsx
Created: Merdon_7.xlsx
Created: Morton_7.xlsx
Created: Newtown_7.xlsx
Created: NorthWaltham_7.xlsx
Created: Otterford_7.xlsx
Created: O

In [12]:
### Export for manual adjustements & fixes
import os
import shutil
from pathlib import Path

src_dir = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\python transcription\script7"
dst_dir = r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\output\OCR 1409-1410\manual adjustement"

# Ensure destination exists
Path(dst_dir).mkdir(parents=True, exist_ok=True)

# HARD STOP if destination already contains files
existing_files = [
    f for f in os.listdir(dst_dir)
    if os.path.isfile(os.path.join(dst_dir, f))
]

if existing_files:
    raise RuntimeError(
        f"Aborting copy: destination folder already contains {len(existing_files)} file(s).\n"
        f"First few files: {existing_files[:5]}"
    )

# Copy + rename
for fname in os.listdir(src_dir):
    if not fname.lower().endswith(".xlsx"):
        continue

    base, ext = os.path.splitext(fname)
    new_name = f"{base}_manual_fix{ext}"

    shutil.copy2(
        os.path.join(src_dir, fname),
        os.path.join(dst_dir, new_name)
    )

print("Copy completed: files copied and renamed with '_manual_fix'.")


Copy completed: files copied and renamed with '_manual_fix'.


In [14]:
from openpyxl import load_workbook
import os

# Apply formulas to the copied "_manual_fix" files in dst_dir
for fname in os.listdir(dst_dir):
    if not fname.lower().endswith(".xlsx"):
        continue
    if "_manual_fix" not in fname:
        continue

    fpath = os.path.join(dst_dir, fname)
    wb = load_workbook(fpath)

    if "Stock" not in wb.sheetnames:
        print(f"Skipping {fname}: no 'stock' sheet")
        continue

    ws = wb["Stock"]

    for row in range(5, 31):  # K5:K30
        ws[f"K{row}"] = f"=B{row}+C{row}-H{row}-I{row}"

    wb.save(fpath)

print("Done: wrote formulas to K5:K30 in all '_manual_fix' files (sheet 'stock').")


Done: wrote formulas to K5:K30 in all '_manual_fix' files (sheet 'stock').
